# MNQ ORB retenue - Notebook final client

Ce notebook reconstruit de bout en bout la sleeve ORB retenue dans le portefeuille MNQ ORB + Pullback.

Ce que contient ce livrable :
- le chargement et le nettoyage des donnees MNQ 1 minute,
- la logique ORB complete visible dans le notebook,
- la reconstruction du signal, du filtrage et du backtest,
- les heatmaps IS/OOS sur chaque famille de parametres importante,
- une cellule de parametrage unique pour challenger facilement la strategie.

In [1]:
import json
import math
import datetime as dt
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Impossible de retrouver la racine du repo depuis le notebook.")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 240)


In [2]:
REPO_ROOT = ROOT
DEFAULT_TIMEZONE = "America/New_York"
RTH_START = "09:30"
RTH_END = "16:00"
ETH_START = "18:00"
ETH_END = "17:00"

INSTRUMENT_SPECS = {
    "NQ": {"tick_size": 0.25, "tick_value_usd": 5.0, "point_value_usd": 20.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "MNQ": {"tick_size": 0.25, "tick_value_usd": 0.5, "point_value_usd": 2.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "ES": {"tick_size": 0.25, "tick_value_usd": 12.5, "point_value_usd": 50.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "MES": {"tick_size": 0.25, "tick_value_usd": 1.25, "point_value_usd": 5.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "RTY": {"tick_size": 0.10, "tick_value_usd": 5.0, "point_value_usd": 50.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "M2K": {"tick_size": 0.10, "tick_value_usd": 0.5, "point_value_usd": 5.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
    "MGC": {"tick_size": 0.10, "tick_value_usd": 1.0, "point_value_usd": 10.0, "commission_per_side_usd": 1.25, "slippage_ticks": 1},
}

DEFAULT_SYMBOL = "MNQ"
DEFAULT_TICK_SIZE = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["tick_size"]
DEFAULT_TICK_VALUE_USD = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["tick_value_usd"]
DEFAULT_POINT_VALUE_USD = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["point_value_usd"]
DEFAULT_COMMISSION_PER_SIDE_USD = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["commission_per_side_usd"]
DEFAULT_SLIPPAGE_TICKS = INSTRUMENT_SPECS[DEFAULT_SYMBOL]["slippage_ticks"]
DEFAULT_INITIAL_CAPITAL_USD = 50_000.0
DEFAULT_MONTHLY_SUBSCRIPTION_COST_USD = 150.0

def get_instrument_spec(symbol: str) -> dict:
    key = symbol.upper()
    if key not in INSTRUMENT_SPECS:
        raise ValueError(f"Unknown instrument {symbol!r}")
    return INSTRUMENT_SPECS[key].copy()

def resolve_dataset_path(explicit_path, symbol: str, timeframe: str = "1m") -> Path:
    if explicit_path is not None:
        return Path(explicit_path)
    files = sorted((REPO_ROOT / "data" / "processed" / "parquet").glob(f"{symbol}_c_0_{timeframe}_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No processed dataset found for {symbol} {timeframe}.")
    return files[-1]

def fmt_money(value):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):,.1f} USD"

def fmt_pct(value, digits=2):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}%"

def fmt_float(value, digits=3):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}"

def parameter_table(rows):
    frame = pd.DataFrame(rows)
    display(frame)
    return frame

def normalize_sessions(sessions):
    return pd.to_datetime(pd.Index(sessions), errors="coerce").normalize().tolist()

def session_set(sessions):
    return set(pd.to_datetime(pd.Index(sessions), errors="coerce").date)

def subset_trades(trades, sessions):
    if trades.empty:
        return trades.copy()
    allowed = session_set(sessions)
    out = trades.copy()
    out["_session_key"] = pd.to_datetime(out["session_date"], errors="coerce").dt.date
    return out.loc[out["_session_key"].isin(allowed)].drop(columns=["_session_key"]).copy().reset_index(drop=True)

def daily_results_from_trades(trades, sessions, initial_capital):
    calendar = pd.DataFrame({"session_date": normalize_sessions(sessions)})
    calendar = calendar.dropna().drop_duplicates().sort_values("session_date").reset_index(drop=True)
    if calendar.empty:
        return pd.DataFrame(columns=["session_date", "daily_pnl_usd", "daily_trade_count", "equity", "peak_equity", "drawdown_usd", "drawdown_pct", "daily_return"])

    if trades.empty:
        daily = calendar.copy()
        daily["daily_pnl_usd"] = 0.0
        daily["daily_trade_count"] = 0
    else:
        view = trades.copy()
        view["session_date"] = pd.to_datetime(view["session_date"], errors="coerce").dt.normalize()
        grouped = (
            view.groupby("session_date", as_index=False)
            .agg(daily_pnl_usd=("net_pnl_usd", "sum"), daily_trade_count=("trade_id", "count"))
        )
        daily = calendar.merge(grouped, on="session_date", how="left")
        daily["daily_pnl_usd"] = pd.to_numeric(daily["daily_pnl_usd"], errors="coerce").fillna(0.0)
        daily["daily_trade_count"] = pd.to_numeric(daily["daily_trade_count"], errors="coerce").fillna(0).astype(int)

    prev_equity = float(initial_capital)
    rets = []
    equities = []
    for pnl in pd.to_numeric(daily["daily_pnl_usd"], errors="coerce").fillna(0.0):
        rets.append(float(pnl) / prev_equity if prev_equity else 0.0)
        prev_equity += float(pnl)
        equities.append(prev_equity)

    daily["daily_return"] = rets
    daily["equity"] = equities
    daily["peak_equity"] = daily["equity"].cummax()
    daily["drawdown_usd"] = daily["equity"] - daily["peak_equity"]
    daily["drawdown_pct"] = np.where(daily["peak_equity"] > 0, (daily["equity"] / daily["peak_equity"] - 1.0) * 100.0, 0.0)
    return daily

def curve_from_returns(session_dates, daily_return, initial_capital, label="curve"):
    out = pd.DataFrame({"session_date": pd.to_datetime(pd.Series(session_dates), errors="coerce").dt.normalize()})
    out["daily_return"] = pd.to_numeric(pd.Series(daily_return), errors="coerce").fillna(0.0)
    if (out["daily_return"] <= -1.0).any():
        worst = float(out["daily_return"].min())
        raise ValueError(f"{label}: daily return <= -100% ({worst:.2%}).")
    out["equity"] = float(initial_capital) * (1.0 + out["daily_return"]).cumprod()
    out["daily_pnl_usd"] = out["equity"].diff().fillna(out["equity"].iloc[0] - float(initial_capital))
    out["peak_equity"] = out["equity"].cummax()
    out["drawdown_usd"] = out["equity"] - out["peak_equity"]
    out["drawdown_pct"] = np.where(out["peak_equity"] > 0, (out["equity"] / out["peak_equity"] - 1.0) * 100.0, 0.0)
    return out

def curve_metrics(curve, trades=None, initial_capital=50_000.0):
    if curve.empty:
        return {
            "net_pnl_usd": 0.0,
            "return_pct": 0.0,
            "cagr_pct": np.nan,
            "sharpe": 0.0,
            "sortino": 0.0,
            "annualized_vol_pct": 0.0,
            "profit_factor": 0.0,
            "max_drawdown_usd": 0.0,
            "max_drawdown_pct": 0.0,
            "worst_day_usd": 0.0,
            "n_trades": 0,
            "win_rate": 0.0,
        }
    ordered = curve.sort_values("session_date").reset_index(drop=True)
    rets = pd.to_numeric(ordered["daily_return"], errors="coerce").fillna(0.0)
    equity = pd.to_numeric(ordered["equity"], errors="coerce").fillna(float(initial_capital))
    pnl = pd.to_numeric(ordered["daily_pnl_usd"], errors="coerce").fillna(0.0)
    start = pd.Timestamp(ordered["session_date"].iloc[0])
    end = pd.Timestamp(ordered["session_date"].iloc[-1])
    years = max(((end - start).days + 1) / 365.25, 1 / 365.25)
    final_equity = float(equity.iloc[-1])
    sharpe = float(rets.mean() / rets.std(ddof=0) * math.sqrt(252.0)) if len(rets) > 1 and rets.std(ddof=0) > 0 else 0.0
    downside = rets[rets < 0]
    downside_std = float(np.sqrt(np.mean(np.square(downside)))) if len(downside) else 0.0
    sortino = float(rets.mean() / downside_std * math.sqrt(252.0)) if downside_std > 0 else 0.0
    cagr = float(((final_equity / float(initial_capital)) ** (1.0 / years) - 1.0) * 100.0) if final_equity > 0 else np.nan
    if trades is not None and not trades.empty:
        trade_pnl = pd.to_numeric(trades["net_pnl_usd"], errors="coerce").fillna(0.0)
        wins = trade_pnl[trade_pnl > 0]
        losses = trade_pnl[trade_pnl < 0]
        profit_factor = float(wins.sum() / abs(losses.sum())) if abs(losses.sum()) > 0 else float("inf")
        n_trades = int(len(trade_pnl))
        win_rate = float((trade_pnl > 0).mean())
    else:
        wins = pnl[pnl > 0]
        losses = pnl[pnl < 0]
        profit_factor = float(wins.sum() / abs(losses.sum())) if abs(losses.sum()) > 0 else float("inf")
        n_trades = 0
        win_rate = np.nan
    return {
        "net_pnl_usd": float(final_equity - float(initial_capital)),
        "return_pct": float((final_equity / float(initial_capital) - 1.0) * 100.0),
        "cagr_pct": cagr,
        "sharpe": sharpe,
        "sortino": sortino,
        "annualized_vol_pct": float(rets.std(ddof=0) * math.sqrt(252.0) * 100.0) if len(rets) > 1 else 0.0,
        "profit_factor": profit_factor,
        "max_drawdown_usd": float(abs(pd.to_numeric(ordered["drawdown_usd"], errors="coerce").min())),
        "max_drawdown_pct": float(abs(pd.to_numeric(ordered["drawdown_pct"], errors="coerce").min())),
        "worst_day_usd": float(pnl.min()),
        "n_trades": n_trades,
        "win_rate": win_rate,
    }

def summarize_strategy_scopes(label, full_curve, full_trades, is_sessions, oos_sessions, initial_capital):
    is_curve = full_curve.loc[full_curve["session_date"].isin(set(normalize_sessions(is_sessions)))].copy()
    oos_trades = subset_trades(full_trades, oos_sessions)
    oos_curve = daily_results_from_trades(oos_trades, oos_sessions, initial_capital)
    rows = [
        {"strategy": label, "scope": "full", **curve_metrics(full_curve, full_trades, initial_capital)},
        {"strategy": label, "scope": "is", **curve_metrics(is_curve, subset_trades(full_trades, is_sessions), initial_capital)},
        {"strategy": label, "scope": "oos", **curve_metrics(oos_curve, oos_trades, initial_capital)},
    ]
    return pd.DataFrame(rows)

def _metric_columns_for_scope(frame, scope, metric):
    mapping = {
        "sharpe": f"{scope}_sharpe_ratio",
        "net_pnl_usd": f"{scope}_net_pnl",
        "profit_factor": f"{scope}_profit_factor",
        "max_drawdown_usd": f"{scope}_max_drawdown",
        "prop_score": f"{scope}_prop_score",
        "n_trades": f"{scope}_nb_trades",
    }
    return mapping[metric]

def plot_scope_heatmap(frame, x, metric, title, text_auto=".2f", color_continuous_scale="RdYlGn"):
    data = frame.copy()
    data[x] = data[x].astype(str)
    pivot = data.pivot_table(index="scope", columns=x, values=metric, aggfunc="mean")
    fig = px.imshow(pivot, aspect="auto", text_auto=text_auto, color_continuous_scale=color_continuous_scale, title=title)
    fig.update_layout(template=PLOT_TEMPLATE, width=1100, height=380)
    fig.update_xaxes(title=x)
    fig.update_yaxes(title="scope")
    fig.show()

def plot_is_oos_heatmaps(frame, x, y, metric, title, text_auto=".2f", color_continuous_scale="RdYlGn"):
    scopes = [("is", "IS"), ("oos", "OOS")]
    fig = make_subplots(rows=1, cols=2, subplot_titles=[label for _, label in scopes], horizontal_spacing=0.10)
    zmin = None
    zmax = None
    if metric not in {"max_drawdown_usd"}:
        finite = pd.to_numeric(frame[metric], errors="coerce")
        if finite.notna().any():
            zmin = float(finite.min())
            zmax = float(finite.max())
    for idx, (scope, label) in enumerate(scopes, start=1):
        scoped = frame.loc[frame["scope"].eq(scope)].copy()
        scoped[x] = scoped[x].astype(str)
        scoped[y] = scoped[y].astype(str)
        pivot = scoped.pivot_table(index=y, columns=x, values=metric, aggfunc="mean").sort_index()
        heatmap = go.Heatmap(
            z=pivot.values,
            x=[str(v) for v in pivot.columns],
            y=[str(v) for v in pivot.index],
            colorscale=color_continuous_scale,
            zmin=zmin,
            zmax=zmax,
            colorbar=dict(title=metric) if idx == 2 else None,
            showscale=idx == 2,
            text=np.round(pivot.values, 2 if metric not in {"net_pnl_usd", "max_drawdown_usd"} else 0),
            texttemplate="%{text}",
        )
        fig.add_trace(heatmap, row=1, col=idx)
    fig.update_layout(template=PLOT_TEMPLATE, width=1350, height=500, title=title)
    fig.update_xaxes(title=x, row=1, col=1)
    fig.update_xaxes(title=x, row=1, col=2)
    fig.update_yaxes(title=y, row=1, col=1)
    fig.update_yaxes(title=y, row=1, col=2)
    fig.show()

def row_to_scope_records(row, extra=None):
    extra = extra or {}
    records = []
    for scope in ("is", "oos"):
        records.append(
            {
                **extra,
                "scope": scope,
                "net_pnl_usd": float(row.get(f"{scope}_net_pnl", 0.0)),
                "sharpe": float(row.get(f"{scope}_sharpe_ratio", 0.0)),
                "profit_factor": float(row.get(f"{scope}_profit_factor", 0.0)),
                "max_drawdown_usd": abs(float(row.get(f"{scope}_max_drawdown", 0.0))),
                "prop_score": float(row.get(f"{scope}_prop_score", 0.0)),
                "n_trades": int(row.get(f"{scope}_nb_trades", 0) or 0),
            }
        )
    return records

def clean_ohlcv(df: pd.DataFrame, drop_duplicate_timestamps: bool = True) -> pd.DataFrame:
    """Sort, type-cast numeric columns, and optionally remove duplicate timestamps."""
    out = df.copy()
    out = out.sort_values("timestamp").reset_index(drop=True)

    if drop_duplicate_timestamps:
        out = out.drop_duplicates(subset=["timestamp"], keep="last")

    numeric_columns = [c for c in ["open", "high", "low", "close", "volume", "open interest"] if c in out.columns]
    for col in numeric_columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=["timestamp", "open", "high", "low", "close", "volume"])
    return out.reset_index(drop=True)

def _normalize_ohlcv_frame(df: pd.DataFrame, timezone: str) -> pd.DataFrame:
    """Normalize columns and timestamps on a dataframe already loaded from disk."""
    out = df.copy()
    out.columns = [col.strip().lower() for col in out.columns]

    if "timestamp" not in out.columns and "ts_event" not in out.columns:
        raise ValueError("Missing required 'timestamp' or 'ts_event' column")

    timestamp_col = "timestamp" if "timestamp" in out.columns else "ts_event"
    out[timestamp_col] = pd.to_datetime(out[timestamp_col], errors="coerce")
    if out[timestamp_col].isna().any():
        raise ValueError(f"Found unparsable {timestamp_col} values")

    if timestamp_col != "timestamp":
        out = out.rename(columns={timestamp_col: "timestamp"})

    if out["timestamp"].dt.tz is None:
        out["timestamp"] = out["timestamp"].dt.tz_localize(timezone)
    else:
        out["timestamp"] = out["timestamp"].dt.tz_convert(timezone)

    return out

def load_ohlcv_csv(path: Path | str, timezone: str = DEFAULT_TIMEZONE) -> pd.DataFrame:
    """Load OHLCV csv with timestamp parsing and normalized lowercase columns.

    Assumptions:
    - timestamps in source csv are US Eastern local times
    - timestamp marks beginning of bar
    - intraday bars may have missing minutes due to zero-volume bars not present
    """
    csv_path = Path(path)
    return _normalize_ohlcv_frame(pd.read_csv(csv_path), timezone=timezone)

def load_ohlcv_file(path: Path | str, timezone: str = DEFAULT_TIMEZONE) -> pd.DataFrame:
    """Load a supported OHLCV file and normalize timestamps/column names."""
    file_path = Path(path)
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return load_ohlcv_csv(file_path, timezone=timezone)
    if suffix == ".parquet":
        df = pd.read_parquet(file_path)
        if df.index.name == 'timestamp' or 'timestamp' not in df.columns:
            df = df.reset_index()
        return _normalize_ohlcv_frame(df, timezone=timezone)

    raise ValueError(f"Unsupported OHLCV file format: {file_path.suffix}")

def _session_mask(timestamps: pd.Series, start_time: str, end_time: str) -> pd.Series:
    """Build boolean mask for a time range, handling overnight windows."""
    times = timestamps.dt.time
    start = dt.time.fromisoformat(start_time)
    end = dt.time.fromisoformat(end_time)
    if start <= end:
        return (times >= start) & (times <= end)
    return (times >= start) | (times <= end)

def filter_session(df: pd.DataFrame, start_time: str, end_time: str) -> pd.DataFrame:
    """Filter bars between start_time and end_time (inclusive), local dataframe timezone."""
    mask = _session_mask(df["timestamp"], start_time, end_time)
    return df.loc[mask].reset_index(drop=True)

def extract_rth(df: pd.DataFrame) -> pd.DataFrame:
    """Extract regular trading hours."""
    return filter_session(df, RTH_START, RTH_END)

def add_session_date(df: pd.DataFrame) -> pd.DataFrame:
    """Add session_date from local timestamp date for robust session grouping."""
    out = df.copy()
    out["session_date"] = out["timestamp"].dt.date
    return out

def build_session_time(session_timestamp: pd.Timestamp, clock_time: str) -> pd.Timestamp:
    """Return the session date at the requested clock time, preserving timezone."""
    time_value = dt.time.fromisoformat(clock_time)
    timestamp = pd.Timestamp(session_timestamp)
    return timestamp.replace(
        hour=time_value.hour,
        minute=time_value.minute,
        second=time_value.second,
        microsecond=0,
        nanosecond=0,
    )

def add_intraday_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add minute-of-day, weekday, and bar rank in session."""
    out = df.copy()
    out["session_date"] = out["timestamp"].dt.date
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["weekday"] = out["timestamp"].dt.dayofweek
    out["bar_in_session"] = out.groupby("session_date").cumcount() + 1
    return out

def add_session_vwap(
    df: pd.DataFrame,
    price_mode: str = "typical",
    price_volume_col: str | None = None,
) -> pd.DataFrame:
    """Add an intraday VWAP column that resets each session."""
    out = df.copy()
    if "session_date" not in out.columns:
        out["session_date"] = out["timestamp"].dt.date

    volume = out["volume"].fillna(0.0)

    if price_volume_col is not None:
        if price_volume_col not in out.columns:
            raise ValueError(f"Missing precomputed price-volume column '{price_volume_col}'.")
        pv = pd.to_numeric(out[price_volume_col], errors="coerce").fillna(0.0)
    elif price_mode == "close":
        price = out["close"]
        pv = price * volume
    elif price_mode == "typical":
        price = (out["high"] + out["low"] + out["close"]) / 3.0
        pv = price * volume
    else:
        raise ValueError("price_mode must be 'close' or 'typical'.")

    cumulative_pv = pv.groupby(out["session_date"]).cumsum()
    cumulative_volume = volume.groupby(out["session_date"]).cumsum()
    out["session_vwap"] = np.where(cumulative_volume > 0, cumulative_pv / cumulative_volume, np.nan)
    return out

def add_continuous_session_vwap(
    df: pd.DataFrame,
    price_mode: str = "typical",
    session_start_hour: int = 18,
    tz: str = "America/New_York",
    timestamp_col: str = "timestamp",
) -> pd.DataFrame:
    """
    Add a futures-style continuous session VWAP.

    The VWAP resets at `session_start_hour` in local timezone, so it includes:
    - overnight
    - premarket
    - regular US session

    Example for CME equity index futures:
    session_start_hour = 18 means the session runs approximately
    from 18:00 ET to 17:59:59 ET next day.
    """
    out = df.copy()

    ts = out[timestamp_col]
    if ts.dt.tz is None:
        ts_local = ts.dt.tz_localize(tz)
    else:
        ts_local = ts.dt.tz_convert(tz)
    out["_ts_local"] = ts_local

    # Build continuous session date by shifting all bars from session_start_hour or later to next calendar day.
    session_date = out["_ts_local"].dt.date
    mask_next_session = out["_ts_local"].dt.hour >= session_start_hour
    session_date = session_date.where(~mask_next_session, (out["_ts_local"] + pd.Timedelta(days=1)).dt.date)
    out["continuous_session_date"] = session_date

    if price_mode == "close":
        price = out["close"]
    elif price_mode == "typical":
        price = (out["high"] + out["low"] + out["close"]) / 3.0
    else:
        raise ValueError("price_mode must be 'close' or 'typical'.")

    volume = out["volume"].fillna(0.0)
    pv = price * volume

    cumulative_pv = pv.groupby(out["continuous_session_date"]).cumsum()
    cumulative_volume = volume.groupby(out["continuous_session_date"]).cumsum()

    out["continuous_session_vwap"] = np.where(
        cumulative_volume > 0,
        cumulative_pv / cumulative_volume,
        np.nan,
    )

    out = out.drop(columns=["_ts_local"])
    return out

def compute_opening_range(
    df: pd.DataFrame,
    or_minutes: int = 30,
    opening_time: str = "09:00:00",
) -> pd.DataFrame:
    """Compute opening range high/low/width/midpoint from the configured opening time."""
    out = df.copy()
    out["session_date"] = out["timestamp"].dt.date

    range_frames = []
    for session_date, group in out.groupby("session_date", sort=True):
        if group.empty:
            continue

        start = build_session_time(group["timestamp"].iloc[0], opening_time)
        end = start + pd.Timedelta(minutes=or_minutes)
        opening_window = group[(group["timestamp"] >= start) & (group["timestamp"] < end)]
        if opening_window.empty:
            continue

        high = opening_window["high"].max()
        low = opening_window["low"].min()
        width = high - low
        midpoint = (high + low) / 2.0
        range_frames.append(
            {
                "session_date": session_date,
                "or_high": high,
                "or_low": low,
                "or_width": width,
                "or_midpoint": midpoint,
            }
        )

    or_df = pd.DataFrame(range_frames)
    return out.merge(or_df, on="session_date", how="left")

def _normalize_windows(window: int | Iterable[int]) -> list[int]:
    if isinstance(window, (int, np.integer)):
        windows = [int(window)]
    else:
        windows = [int(value) for value in window]

    clean = sorted({value for value in windows if value > 0})
    if not clean:
        raise ValueError("ATR window must contain at least one positive integer.")
    return clean

def add_atr(df: pd.DataFrame, window: int | Iterable[int] = 14) -> pd.DataFrame:
    """Add simplified ATR feature(s) for one or many rolling windows."""
    out = df.copy()
    prev_close = out["close"].shift(1)
    tr = np.maximum(
        out["high"] - out["low"],
        np.maximum((out["high"] - prev_close).abs(), (out["low"] - prev_close).abs()),
    )
    for w in _normalize_windows(window):
        out[f"atr_{w}"] = tr.rolling(w).mean()
    return out

TRADE_LOG_COLUMNS = [
    "trade_id",
    "session_date",
    "direction",
    "quantity",
    "entry_time",
    "entry_price",
    "stop_price",
    "target_price",
    "exit_time",
    "exit_price",
    "exit_reason",
    "account_size_usd",
    "risk_per_trade_pct",
    "risk_budget_usd",
    "risk_per_contract_usd",
    "actual_risk_usd",
    "trade_risk_usd",
    "notional_usd",
    "leverage_used",
    "pnl_points",
    "pnl_ticks",
    "pnl_usd",
    "fees",
    "net_pnl_usd",
]

def empty_trade_log() -> pd.DataFrame:
    """Return an empty trade log dataframe with standard columns."""
    return pd.DataFrame(columns=TRADE_LOG_COLUMNS)

def trade_to_record(trade_id: int, data: dict[str, Any]) -> dict[str, Any]:
    """Build standardized trade record dict."""
    row = {col: None for col in TRADE_LOG_COLUMNS}
    row["trade_id"] = trade_id
    row.update(data)
    return row

@dataclass
class ExecutionModel:
    """Simple fixed commission + fixed slippage model."""

    commission_per_side_usd: float = DEFAULT_COMMISSION_PER_SIDE_USD
    slippage_ticks: float = float(DEFAULT_SLIPPAGE_TICKS)
    tick_size: float = DEFAULT_TICK_SIZE

    def apply_slippage(self, price: float, direction: int, is_entry: bool) -> float:
        """Adjust price by slippage in unfavorable direction."""
        slippage_points = self.slippage_ticks * self.tick_size
        if direction == 1:
            return price + slippage_points if is_entry else price - slippage_points
        return price - slippage_points if is_entry else price + slippage_points

    def round_trip_fees(self, quantity: int = 1) -> float:
        """Return total round-trip fees in USD for the requested quantity."""
        return 2.0 * self.commission_per_side_usd * quantity

EQUITY_TICK_SIZE = 0.01

EQUITY_POINT_VALUE_USD = 1.0

EQUITY_PAPER_COMMISSION_PER_SHARE = 0.0005

@dataclass(frozen=True)
class InstrumentDetails:
    """Execution metadata inferred from the selected dataset symbol."""

    symbol: str
    asset_class: str
    tick_size: float
    tick_value_usd: float
    point_value_usd: float
    commission_per_side_usd: float
    slippage_ticks: int

def resolve_instrument_details(symbol: str) -> InstrumentDetails:
    """Resolve instrument specifications from repo defaults or equity-like fallback."""
    upper_symbol = symbol.upper()
    if upper_symbol in INSTRUMENT_SPECS:
        spec = INSTRUMENT_SPECS[upper_symbol]
        return InstrumentDetails(
            symbol=upper_symbol,
            asset_class="futures",
            tick_size=float(spec["tick_size"]),
            tick_value_usd=float(spec["tick_value_usd"]),
            point_value_usd=float(spec["point_value_usd"]),
            commission_per_side_usd=float(spec["commission_per_side_usd"]),
            slippage_ticks=int(spec["slippage_ticks"]),
        )

    return InstrumentDetails(
        symbol=upper_symbol,
        asset_class="equity",
        tick_size=EQUITY_TICK_SIZE,
        tick_value_usd=EQUITY_TICK_SIZE * EQUITY_POINT_VALUE_USD,
        point_value_usd=EQUITY_POINT_VALUE_USD,
        commission_per_side_usd=EQUITY_PAPER_COMMISSION_PER_SHARE,
        slippage_ticks=0,
    )

def build_execution_model_for_profile(
    symbol: str,
    profile_name: str,
    cost_multiplier: float = 1.0,
) -> tuple[ExecutionModel, InstrumentDetails]:
    """Build the execution model used by a VWAP run."""
    details = resolve_instrument_details(symbol)
    if cost_multiplier <= 0:
        raise ValueError("cost_multiplier must be strictly positive.")

    if profile_name == "repo_realistic":
        commission = details.commission_per_side_usd * cost_multiplier
        slippage_ticks = int(round(details.slippage_ticks * cost_multiplier))
        return (
            ExecutionModel(
                commission_per_side_usd=commission,
                slippage_ticks=slippage_ticks,
                tick_size=details.tick_size,
            ),
            details,
        )

    if profile_name == "paper_reference":
        if details.asset_class == "futures":
            commission = details.commission_per_side_usd * cost_multiplier
            slippage_ticks = int(round(details.slippage_ticks * cost_multiplier))
            return (
                ExecutionModel(
                    commission_per_side_usd=commission,
                    slippage_ticks=slippage_ticks,
                    tick_size=details.tick_size,
                ),
                details,
            )
        return (
            ExecutionModel(
                commission_per_side_usd=EQUITY_PAPER_COMMISSION_PER_SHARE * cost_multiplier,
                slippage_ticks=0,
                tick_size=details.tick_size,
            ),
            details,
        )

    raise ValueError(f"Unsupported execution profile '{profile_name}'.")

def latest_path_for_symbol(symbol: str, input_paths: dict[str, Path] | None = None) -> Path:
    if input_paths is not None and symbol in input_paths:
        return Path(input_paths[symbol])
    files = sorted((REPO_ROOT / "data" / "processed" / "parquet").glob(f"{symbol}_c_0_1m_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No input dataset found for {symbol} in data/processed/parquet.")
    return files[-1]

def load_symbol_data(symbol: str, input_paths: dict[str, Path] | None = None) -> pd.DataFrame:
    source_path = latest_path_for_symbol(symbol, input_paths=input_paths)
    return clean_ohlcv(load_ohlcv_file(source_path))

def resample_rth_1h(raw: pd.DataFrame) -> pd.DataFrame:
    scoped = extract_rth(raw.copy())
    if scoped.empty:
        return scoped
    scoped["timestamp"] = pd.to_datetime(scoped["timestamp"], errors="coerce")
    scoped = scoped.set_index("timestamp").sort_index()
    bars = scoped.resample("1h", label="left", closed="left", origin="start_day", offset="30min").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    return bars.dropna(subset=["open", "high", "low", "close"]).reset_index()

def split_sessions(frame: pd.DataFrame, ratio: float = 0.7) -> tuple[list, list]:
    sessions = sorted(pd.to_datetime(frame["session_date"]).dt.date.unique())
    if len(sessions) < 2:
        raise ValueError("Need at least two sessions to build an IS/OOS split.")
    cut = max(1, int(len(sessions) * ratio))
    cut = min(cut, len(sessions) - 1)
    return sessions[:cut], sessions[cut:]


In [3]:
@dataclass(frozen=True)
class PropConstraintConfig:
    """Topstep-style research constraints used for practical ranking."""

    name: str = "topstep_50k_reference"
    account_size_usd: float = DEFAULT_INITIAL_CAPITAL_USD
    max_loss_limit_usd: float = 2_000.0
    daily_loss_limit_usd: float | None = 1_000.0
    profit_target_usd: float = 3_000.0
    monthly_subscription_cost_usd: float = DEFAULT_MONTHLY_SUBSCRIPTION_COST_USD
    trading_days_per_month: float = 21.0
    daily_loss_limit_basis: str = "realized_daily_pnl"

def _loss_streak_lengths(pnl: pd.Series) -> list[int]:
    """Return lengths of consecutive losing-trade streaks."""
    streaks: list[int] = []
    current = 0

    for value in pnl:
        if value < 0:
            current += 1
            continue
        if current > 0:
            streaks.append(current)
            current = 0

    if current > 0:
        streaks.append(current)
    return streaks

def _resolve_session_dates(
    trades: pd.DataFrame,
    signal_df: pd.DataFrame | None,
    session_dates: pd.Series | pd.Index | list | None,
) -> pd.Index:
    """Resolve the session-date index used for daily metrics."""
    if session_dates is not None:
        return pd.Index(pd.to_datetime(pd.Index(session_dates)).date)
    if signal_df is not None and "session_date" in signal_df.columns:
        return pd.Index(pd.to_datetime(signal_df["session_date"]).dt.date.unique())
    if not trades.empty and "session_date" in trades.columns:
        return pd.Index(pd.to_datetime(trades["session_date"]).dt.date.unique())
    return pd.Index([])

def _daily_pnl(trades: pd.DataFrame, resolved_session_dates: pd.Index) -> pd.Series:
    """Aggregate daily PnL, including flat days when session dates are provided."""
    if trades.empty:
        if resolved_session_dates.empty:
            return pd.Series(dtype=float)
        return pd.Series(0.0, index=resolved_session_dates, dtype=float)

    daily = trades.groupby("session_date")["net_pnl_usd"].sum()
    daily.index = pd.Index(pd.to_datetime(daily.index).date)
    if resolved_session_dates.empty:
        return daily.sort_index()
    return daily.reindex(resolved_session_dates, fill_value=0.0).sort_index()

def _prop_empty_metrics(
    resolved_session_dates: pd.Index,
    capital: float,
    prop_constraints: PropConstraintConfig | None,
) -> dict[str, float | int | bool | str]:
    """Return prop-style defaults for empty trade logs."""
    if prop_constraints is None:
        return {}

    observed_months = (
        float(len(resolved_session_dates) / prop_constraints.trading_days_per_month)
        if prop_constraints.trading_days_per_month > 0
        else 0.0
    )
    subscription_drag = observed_months * prop_constraints.monthly_subscription_cost_usd

    return {
        "prop_constraint_profile": prop_constraints.name,
        "profit_target_usd": prop_constraints.profit_target_usd,
        "max_loss_limit_usd": prop_constraints.max_loss_limit_usd,
        "daily_loss_limit_usd": prop_constraints.daily_loss_limit_usd or 0.0,
        "daily_loss_limit_active": bool(prop_constraints.daily_loss_limit_usd is not None),
        "profit_target_reached": False,
        "days_to_profit_target": np.nan,
        "days_to_reach_3000_profit_target": np.nan,
        "estimated_months_to_profit_target": np.nan,
        "profit_target_reached_before_max_loss": False,
        "breaches_max_loss_limit": False,
        "max_loss_limit_buffer_usd": prop_constraints.max_loss_limit_usd,
        "any_daily_loss_limit_breach": False,
        "number_of_daily_loss_limit_breaches": 0,
        "minimum_equity_before_target_hit": np.nan,
        "minimum_equity_observed": capital,
        "max_adverse_drawdown_before_target_hit": np.nan,
        "subscription_drag_estimate": float(subscription_drag),
        "estimated_pnl_after_subscription": float(-subscription_drag),
        "profit_target_progress_pct": 0.0,
    }

def _compute_prop_metrics(
    trades: pd.DataFrame,
    daily: pd.Series,
    resolved_session_dates: pd.Index,
    capital: float,
    prop_constraints: PropConstraintConfig | None,
) -> dict[str, float | int | bool | str]:
    """Estimate practical prop-style survival metrics from the realized path."""
    if prop_constraints is None:
        return {}

    if trades.empty:
        return _prop_empty_metrics(resolved_session_dates, capital, prop_constraints)

    ordered = trades.sort_values("exit_time").reset_index(drop=True).copy()
    ordered["cum_pnl"] = ordered["net_pnl_usd"].astype(float).cumsum()
    ordered["equity"] = capital + ordered["cum_pnl"]

    cumulative = ordered["cum_pnl"]
    minimum_equity_observed = float(ordered["equity"].min())
    worst_cumulative_pnl = float(cumulative.min())

    target_positions = np.flatnonzero(cumulative >= prop_constraints.profit_target_usd)
    target_position = int(target_positions[0]) if len(target_positions) > 0 else None
    profit_target_reached = target_position is not None

    max_loss_positions = np.flatnonzero(cumulative <= -prop_constraints.max_loss_limit_usd)
    max_loss_position = int(max_loss_positions[0]) if len(max_loss_positions) > 0 else None
    breaches_max_loss_limit = max_loss_position is not None

    days_to_profit_target = np.nan
    estimated_months_to_profit_target = np.nan
    minimum_equity_before_target_hit = np.nan
    max_adverse_drawdown_before_target_hit = np.nan

    if profit_target_reached:
        target_session_date = pd.to_datetime(ordered.loc[target_position, "session_date"]).date()
        if not resolved_session_dates.empty:
            days_to_profit_target = int((resolved_session_dates <= target_session_date).sum())
        else:
            days_to_profit_target = int(
                pd.Index(pd.to_datetime(ordered.loc[:target_position, "session_date"]).date).nunique()
            )
        if prop_constraints.trading_days_per_month > 0:
            estimated_months_to_profit_target = float(
                days_to_profit_target / prop_constraints.trading_days_per_month
            )

        path_before_target = ordered.loc[:target_position]
        minimum_equity_before_target_hit = float(path_before_target["equity"].min())
        max_adverse_drawdown_before_target_hit = float(capital - minimum_equity_before_target_hit)

    daily_loss_limit = prop_constraints.daily_loss_limit_usd
    if daily_loss_limit is not None:
        daily_breach_mask = daily <= -daily_loss_limit
        number_of_daily_breaches = int(daily_breach_mask.sum())
    else:
        number_of_daily_breaches = 0
    any_daily_loss_limit_breach = bool(number_of_daily_breaches > 0)

    observed_months = (
        float(len(resolved_session_dates) / prop_constraints.trading_days_per_month)
        if prop_constraints.trading_days_per_month > 0
        else 0.0
    )
    subscription_months = (
        float(estimated_months_to_profit_target) if profit_target_reached else observed_months
    )
    subscription_drag = float(subscription_months * prop_constraints.monthly_subscription_cost_usd)

    return {
        "prop_constraint_profile": prop_constraints.name,
        "profit_target_usd": prop_constraints.profit_target_usd,
        "max_loss_limit_usd": prop_constraints.max_loss_limit_usd,
        "daily_loss_limit_usd": prop_constraints.daily_loss_limit_usd or 0.0,
        "daily_loss_limit_active": bool(prop_constraints.daily_loss_limit_usd is not None),
        "profit_target_reached": bool(profit_target_reached),
        "days_to_profit_target": days_to_profit_target,
        "days_to_reach_3000_profit_target": days_to_profit_target,
        "estimated_months_to_profit_target": estimated_months_to_profit_target,
        "profit_target_reached_before_max_loss": bool(
            profit_target_reached and (max_loss_position is None or target_position <= max_loss_position)
        ),
        "breaches_max_loss_limit": bool(breaches_max_loss_limit),
        "max_loss_limit_buffer_usd": float(prop_constraints.max_loss_limit_usd - abs(min(worst_cumulative_pnl, 0.0))),
        "any_daily_loss_limit_breach": any_daily_loss_limit_breach,
        "number_of_daily_loss_limit_breaches": number_of_daily_breaches,
        "minimum_equity_before_target_hit": minimum_equity_before_target_hit,
        "minimum_equity_observed": minimum_equity_observed,
        "max_adverse_drawdown_before_target_hit": max_adverse_drawdown_before_target_hit,
        "subscription_drag_estimate": subscription_drag,
        "estimated_pnl_after_subscription": float(cumulative.iloc[-1] - subscription_drag),
        "profit_target_progress_pct": float(cumulative.iloc[-1] / prop_constraints.profit_target_usd)
        if prop_constraints.profit_target_usd > 0
        else 0.0,
    }

def compute_metrics(
    trades: pd.DataFrame,
    signal_df: pd.DataFrame | None = None,
    session_dates: pd.Series | pd.Index | list | None = None,
    initial_capital: float | None = None,
    prop_constraints: PropConstraintConfig | None = None,
) -> dict[str, float | int | bool | str]:
    """Compute key strategy metrics from trade log."""
    resolved_session_dates = _resolve_session_dates(trades, signal_df, session_dates)
    n_sessions = int(len(resolved_session_dates))

    if trades.empty:
        empty_metrics: dict[str, float | int | bool | str] = {
            "n_trades": 0,
            "win_rate": 0.0,
            "avg_win": 0.0,
            "avg_loss": 0.0,
            "average_losing_trade": 0.0,
            "expectancy": 0.0,
            "profit_factor": 0.0,
            "cumulative_pnl": 0.0,
            "max_drawdown": 0.0,
            "max_drawdown_pct": 0.0,
            "sharpe_ratio": 0.0,
            "max_consecutive_losses": 0,
            "avg_consecutive_losses": 0.0,
            "longest_loss_streak": 0,
            "number_of_loss_streaks_2_plus": 0,
            "worst_trade": 0.0,
            "worst_day": 0.0,
            "stop_hit_rate": 0.0,
            "target_hit_rate": 0.0,
            "eod_exit_rate": 0.0,
            "proportion_filtered_out": 0.0,
            "trade_count_after_filters": 0,
            "percent_of_days_traded": 0.0,
            "percent_days_traded": 0.0,
            "avg_R": 0.0,
            "median_R": 0.0,
            "pnl_to_risk_ratio": 0.0,
            "average_loss_streak_length": 0.0,
            "count_loss_streaks_2_plus": 0,
            "daily_sharpe_basis": "daily_pnl_over_static_capital",
            "n_sessions": n_sessions,
            "raw_signal_count": 0,
        }
        empty_metrics.update(_prop_empty_metrics(resolved_session_dates, initial_capital or DEFAULT_INITIAL_CAPITAL_USD, prop_constraints))
        return empty_metrics

    pnl = trades["net_pnl_usd"].astype(float)
    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]

    cumulative = pnl.cumsum()
    drawdown = cumulative - cumulative.cummax()

    gross_profit = wins.sum()
    gross_loss_abs = abs(losses.sum())
    profit_factor = float(gross_profit / gross_loss_abs) if gross_loss_abs > 0 else np.inf

    capital = initial_capital
    if capital is None:
        account_sizes = trades["account_size_usd"].dropna() if "account_size_usd" in trades.columns else pd.Series()
        capital = float(account_sizes.iloc[0]) if not account_sizes.empty else DEFAULT_INITIAL_CAPITAL_USD

    daily = _daily_pnl(trades, resolved_session_dates)
    if len(daily) > 1 and capital > 0:
        daily_returns = daily / capital
        daily_std = daily_returns.std(ddof=0)
        sharpe = (daily_returns.mean() / daily_std) * math.sqrt(252.0) if daily_std > 0 else 0.0
    else:
        sharpe = 0.0

    loss_streaks = _loss_streak_lengths(pnl)
    longest_loss_streak = max(loss_streaks, default=0)
    avg_loss_streak = float(np.mean(loss_streaks)) if loss_streaks else 0.0
    streaks_2_plus = int(sum(length >= 2 for length in loss_streaks))

    raw_signal_count = 0
    filtered_out_count = 0
    if signal_df is not None and "raw_signal" in signal_df.columns:
        raw_signal_count = int(signal_df["raw_signal"].ne(0).sum())
        filtered_out_count = int((signal_df["raw_signal"].ne(0) & signal_df["signal"].eq(0)).sum())

    trade_risk = trades["trade_risk_usd"] if "trade_risk_usd" in trades.columns else pd.Series(dtype=float)
    valid_trade_risk = trade_risk[(trade_risk.notna()) & (trade_risk > 0)]
    if len(valid_trade_risk) == len(trades):
        r_multiple = pnl / valid_trade_risk
        avg_r = float(r_multiple.mean())
        median_r = float(r_multiple.median())
        pnl_to_risk = float(pnl.sum() / valid_trade_risk.sum()) if valid_trade_risk.sum() > 0 else 0.0
    else:
        avg_r = 0.0
        median_r = 0.0
        pnl_to_risk = 0.0

    traded_days = int(pd.Index(pd.to_datetime(trades["session_date"]).dt.date).nunique())
    percent_of_days_traded = float(traded_days / n_sessions) if n_sessions > 0 else 0.0

    time_exit_trades = trades[trades["exit_reason"] == "time_exit"]
    time_exit_win = time_exit_trades["net_pnl_usd"] > 0
    time_exit_loss = time_exit_trades["net_pnl_usd"] < 0
    time_exit_count = int(len(time_exit_trades))
    time_exit_win_rate = float(time_exit_win.mean()) if time_exit_count > 0 else 0.0
    time_exit_loss_rate = float(time_exit_loss.mean()) if time_exit_count > 0 else 0.0
    time_exit_pnl = float(time_exit_trades["net_pnl_usd"].sum()) if time_exit_count > 0 else 0.0

    metrics: dict[str, float | int | bool | str] = {
        "n_trades": int(len(trades)),
        "win_rate": float((pnl > 0).mean()),
        "avg_win": float(wins.mean()) if not wins.empty else 0.0,
        "avg_loss": float(losses.mean()) if not losses.empty else 0.0,
        "average_losing_trade": float(losses.mean()) if not losses.empty else 0.0,
        "expectancy": float(pnl.mean()),
        "profit_factor": float(profit_factor),
        "cumulative_pnl": float(pnl.sum()),
        "max_drawdown": float(drawdown.min()),
        "max_drawdown_pct": float(abs(drawdown.min()) / capital) if capital > 0 else 0.0,
        "sharpe_ratio": float(sharpe),
        "max_consecutive_losses": int(longest_loss_streak),
        "avg_consecutive_losses": avg_loss_streak,
        "longest_loss_streak": int(longest_loss_streak),
        "number_of_loss_streaks_2_plus": streaks_2_plus,
        "worst_trade": float(pnl.min()),
        "worst_day": float(daily.min()) if not daily.empty else 0.0,
        "stop_hit_rate": float((trades["exit_reason"] == "stop").mean()),
        "target_hit_rate": float((trades["exit_reason"] == "target").mean()),
        "time_exit_rate": float((trades["exit_reason"] == "time_exit").mean()),
        "time_exit_count": time_exit_count,
        "time_exit_win_rate": time_exit_win_rate,
        "time_exit_loss_rate": time_exit_loss_rate,
        "time_exit_pnl_usd": time_exit_pnl,
        "eod_exit_rate": float(trades["exit_reason"].isin(["eod_exit"]).mean()),
        "proportion_filtered_out": float(filtered_out_count / raw_signal_count) if raw_signal_count > 0 else 0.0,
        "trade_count_after_filters": int(len(trades)),
        "percent_of_days_traded": percent_of_days_traded,
        "percent_days_traded": percent_of_days_traded,
        "avg_R": avg_r,
        "median_R": median_r,
        "pnl_to_risk_ratio": pnl_to_risk,
        "average_loss_streak_length": avg_loss_streak,
        "count_loss_streaks_2_plus": streaks_2_plus,
        "daily_sharpe_basis": "daily_pnl_over_static_capital",
        "n_sessions": n_sessions,
        "raw_signal_count": raw_signal_count,
    }
    metrics.update(
        _compute_prop_metrics(
            trades=trades,
            daily=daily,
            resolved_session_dates=resolved_session_dates,
            capital=capital,
            prop_constraints=prop_constraints,
        )
    )
    return metrics

@dataclass(frozen=True)
class BaselineEntryConfig:
    """Baseline ORB entry settings (must remain backward compatible)."""

    or_minutes: int = 15
    opening_time: str = "09:30:00"
    direction: str = "long"
    one_trade_per_day: bool = True
    entry_buffer_ticks: int = 2
    stop_buffer_ticks: int = 2
    target_multiple: float = 2.0
    vwap_confirmation: bool = True
    vwap_column: str = "continuous_session_vwap"
    time_exit: str = "16:00:00"
    account_size_usd: float = 50_000.0
    risk_per_trade_pct: float = 0.5
    tick_size: float = 0.25
    entry_on_next_open: bool = True

@dataclass(frozen=True)
class BaselineEnsembleConfig:
    """ATR-ensemble day-selection settings layered on top of entry signals."""

    atr_window: int = 14
    q_lows_pct: tuple[int, ...] = (20, 25, 30)
    q_highs_pct: tuple[int, ...] = (90, 95)
    vote_threshold: float = 0.5

@dataclass(frozen=True)
class CompressionConfig:
    """Compression overlay configuration."""

    mode: str = "none"
    usage: str = "hard_filter"  # hard_filter | soft_vote_bonus
    soft_bonus_votes: float = 1.0

@dataclass(frozen=True)
class ExitConfig:
    """Exit/trailing overlay configuration."""

    mode: str = "baseline"
    force_exit_time: str | None = None
    stagnation_bars: int | None = None
    stagnation_min_r_multiple: float = 0.15
    partial_fraction: float = 0.5

@dataclass(frozen=True)
class DynamicThresholdConfig:
    """Dynamic breakout/noise-gate configuration."""

    mode: str = "disabled"
    noise_lookback: int = 14
    noise_vm: float = 1.0
    threshold_style: str = "max_or_high_noise"  # max_or_high_noise | or_high_plus_k_noise_abs
    noise_k: float = 0.0
    atr_k: float = 0.0
    confirm_bars: int = 1
    schedule: str = "continuous_on_bar_close"  # continuous_on_bar_close | every_5m | every_15m

@dataclass(frozen=True)
class ExperimentConfig:
    """Full experiment definition used by the campaign executor."""

    name: str
    stage: str
    family: str
    baseline_entry: BaselineEntryConfig
    baseline_ensemble: BaselineEnsembleConfig
    compression: CompressionConfig
    exit: ExitConfig
    dynamic_threshold: DynamicThresholdConfig

@dataclass
class CampaignContext:
    """Runtime cache shared by experiment evaluators."""

    all_sessions: list
    is_sessions: list
    oos_sessions: list
    minute_df: object
    candidate_base_df: object
    daily_patterns: object
    noise_cache: dict[int, object] = field(default_factory=dict)

_SIDE_MODE_ALIASES = {
    "both": "both",
    "long": "long",
    "short": "short",
    "long_only": "long",
    "short_only": "short",
}

_DIRECTION_FILTER_MODES = {"none", "vwap_only", "ema_only", "vwap_and_ema"}

def _normalize_side_mode(side_mode: str) -> str:
    """Normalize supported side-mode aliases."""
    normalized = _SIDE_MODE_ALIASES.get(side_mode)
    if normalized is None:
        raise ValueError("direction must be one of both, long, short, long_only, short_only.")
    return normalized

@dataclass
class ORBStrategy:
    """Simple ORB signal generator with optional regime and trend filters."""

    or_minutes: int = 30
    direction: str = "both"
    one_trade_per_day: bool = True
    entry_buffer_ticks: int = 0
    stop_buffer_ticks: int = 0
    target_multiple: float = 1.5
    opening_time: str = "09:00:00"
    time_exit: str = "15:55"
    account_size_usd: float | None = None
    risk_per_trade_pct: float | None = None
    tick_size: float = DEFAULT_TICK_SIZE
    atr_period: int | None = None
    atr_min: float | None = None
    atr_max: float | None = None
    atr_regime: str = "none"
    direction_filter_mode: str = "none"
    vwap_confirmation: bool = False
    vwap_min_distance_ticks: int = 0
    vwap_column: str = "session_vwap"
    ema_length: int | None = None
    filter_price_col: str = "close"

    def _atr_column(self) -> str | None:
        if self.atr_period is None:
            return None
        return f"atr_{self.atr_period}"

    def _passes_atr_filter(self, row: pd.Series) -> bool:
        atr_col = self._atr_column()
        if self.atr_regime == "none" or atr_col is None:
            return True
        if atr_col not in row.index:
            raise ValueError(f"Missing ATR column '{atr_col}' required by the strategy.")

        atr_value = row[atr_col]
        if pd.isna(atr_value):
            return False
        if self.atr_min is not None and atr_value < self.atr_min:
            return False
        if self.atr_max is not None and atr_value > self.atr_max:
            return False
        return True

    def _ensure_ema_column(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.ema_length is None:
            return df

        ema_col = f"ema_{self.ema_length}"
        if ema_col not in df.columns:
            if self.filter_price_col not in df.columns:
                raise ValueError(
                    f"Missing filter price column '{self.filter_price_col}' required for EMA calculation."
                )
            df = df.copy()
            df[ema_col] = df[self.filter_price_col].ewm(span=self.ema_length, adjust=False).mean()
        return df

    def _effective_direction_filter_mode(self) -> str:
        mode = self.direction_filter_mode
        if mode not in _DIRECTION_FILTER_MODES:
            raise ValueError(
                "direction_filter_mode must be one of none, vwap_only, ema_only, vwap_and_ema."
            )

        if not self.vwap_confirmation:
            return mode

        if mode == "none":
            return "vwap_only"
        if mode == "ema_only":
            return "vwap_and_ema"
        return mode

    def _vwap_confirmation_buffer(self) -> float:
        if self.vwap_min_distance_ticks < 0:
            raise ValueError("vwap_min_distance_ticks must be >= 0.")
        return self.vwap_min_distance_ticks * self.tick_size

    def _passes_direction_filter(self, row: pd.Series, signal: int) -> bool:
        mode = self._effective_direction_filter_mode()
        if mode == "none":
            return True

        price = row.get(self.filter_price_col)
        if pd.isna(price):
            return False

        comparisons: list[tuple[str, float, float]] = []
        if mode in ("vwap_only", "vwap_and_ema"):
            if self.vwap_column not in row.index:
                raise ValueError(f"Missing '{self.vwap_column}' column required by the strategy.")
            comparisons.append((self.vwap_column, row[self.vwap_column], self._vwap_confirmation_buffer()))

        if mode in ("ema_only", "vwap_and_ema"):
            if self.ema_length is None:
                raise ValueError("ema_length must be provided when an EMA filter is enabled.")
            ema_col = f"ema_{self.ema_length}"
            if ema_col not in row.index:
                raise ValueError(f"Missing EMA column '{ema_col}' required by the strategy.")
            comparisons.append((ema_col, row[ema_col], 0.0))

        for _, reference_price, min_gap in comparisons:
            if pd.isna(reference_price):
                return False
            if signal == 1 and price <= reference_price + min_gap:
                return False
            if signal == -1 and price >= reference_price - min_gap:
                return False
        return True

    def generate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Generate ORB entry signals based on OR high/low breaks."""
        out = self._ensure_ema_column(df.copy())
        out["signal"] = 0
        out["raw_signal"] = 0
        out["atr_filter_pass"] = True
        out["direction_filter_pass"] = True
        out["filter_pass"] = True
        out["filtered_out"] = False

        buffer = self.entry_buffer_ticks * self.tick_size
        side_mode = _normalize_side_mode(self.direction)

        for _, session_df in out.groupby("session_date", sort=True):
            if session_df.empty:
                continue

            session_df = session_df.sort_values("timestamp")
            session_start = build_session_time(session_df["timestamp"].iloc[0], self.opening_time)
            or_expiry = session_start + pd.Timedelta(minutes=self.or_minutes)
            eligible = session_df["timestamp"] >= or_expiry
            valid_or = session_df["or_high"].notna() & session_df["or_low"].notna()
            long_break = eligible & valid_or & (session_df["close"] >= session_df["or_high"] + buffer)
            short_break = eligible & valid_or & (session_df["close"] <= session_df["or_low"] - buffer)

            raw_signal = pd.Series(0, index=session_df.index, dtype=int)
            if side_mode in ("both", "long"):
                raw_signal.loc[long_break] = 1
            if side_mode in ("both", "short"):
                short_mask = short_break & raw_signal.eq(0)
                raw_signal.loc[short_mask] = -1

            candidate_mask = raw_signal.ne(0)
            if not candidate_mask.any():
                continue

            atr_pass = pd.Series(True, index=session_df.index, dtype=bool)
            atr_col = self._atr_column()
            if self.atr_regime != "none" and atr_col is not None:
                if atr_col not in session_df.columns:
                    raise ValueError(f"Missing ATR column '{atr_col}' required by the strategy.")
                atr_values = session_df[atr_col]
                atr_pass = atr_values.notna()
                if self.atr_min is not None:
                    atr_pass &= atr_values >= self.atr_min
                if self.atr_max is not None:
                    atr_pass &= atr_values <= self.atr_max

            direction_pass = pd.Series(True, index=session_df.index, dtype=bool)
            direction_filter_mode = self._effective_direction_filter_mode()
            if direction_filter_mode != "none":
                if self.filter_price_col not in session_df.columns:
                    raise ValueError(
                        f"Missing filter price column '{self.filter_price_col}' required by the strategy."
                    )

                price = session_df[self.filter_price_col]
                direction_pass = price.notna()
                reference_columns: list[tuple[str, float]] = []

                if direction_filter_mode in ("vwap_only", "vwap_and_ema"):
                    if self.vwap_column not in session_df.columns:
                        raise ValueError(f"Missing '{self.vwap_column}' column required by the strategy.")
                    reference_columns.append((self.vwap_column, self._vwap_confirmation_buffer()))

                if direction_filter_mode in ("ema_only", "vwap_and_ema"):
                    if self.ema_length is None:
                        raise ValueError("ema_length must be provided when an EMA filter is enabled.")
                    ema_col = f"ema_{self.ema_length}"
                    if ema_col not in session_df.columns:
                        raise ValueError(f"Missing EMA column '{ema_col}' required by the strategy.")
                    reference_columns.append((ema_col, 0.0))

                for ref_col, min_gap in reference_columns:
                    reference_price = session_df[ref_col]
                    valid_reference = reference_price.notna()
                    long_ok = raw_signal.ne(1) | (valid_reference & (price > reference_price + min_gap))
                    short_ok = raw_signal.ne(-1) | (valid_reference & (price < reference_price - min_gap))
                    direction_pass &= long_ok & short_ok

            filter_pass = candidate_mask & atr_pass & direction_pass
            considered_index = session_df.index
            if self.one_trade_per_day and filter_pass.any():
                first_signal_idx = filter_pass[filter_pass].index[0]
                cutoff_position = session_df.index.get_loc(first_signal_idx)
                considered_index = session_df.index[: cutoff_position + 1]

            considered_candidates = raw_signal.loc[considered_index].ne(0)
            candidate_index = raw_signal.loc[considered_index][considered_candidates].index
            if len(candidate_index) == 0:
                continue

            out.loc[candidate_index, "raw_signal"] = raw_signal.loc[candidate_index]
            out.loc[candidate_index, "atr_filter_pass"] = atr_pass.loc[candidate_index]
            out.loc[candidate_index, "direction_filter_pass"] = direction_pass.loc[candidate_index]
            out.loc[candidate_index, "filter_pass"] = filter_pass.loc[candidate_index]
            out.loc[candidate_index, "filtered_out"] = ~filter_pass.loc[candidate_index]

            if self.one_trade_per_day:
                passing_candidates = filter_pass.loc[candidate_index]
                if passing_candidates.any():
                    signal_index = passing_candidates[passing_candidates].index[0]
                    out.at[signal_index, "signal"] = int(raw_signal.at[signal_index])
            else:
                passing_index = filter_pass.loc[candidate_index][filter_pass.loc[candidate_index]].index
                out.loc[passing_index, "signal"] = raw_signal.loc[passing_index]

        return out

def _validate_risk_inputs(account_size_usd: float | None, risk_per_trade_pct: float | None) -> None:
    """Validate optional risk-based sizing inputs."""
    if account_size_usd is None and risk_per_trade_pct is None:
        return
    if account_size_usd is None or risk_per_trade_pct is None:
        raise ValueError("account_size_usd and risk_per_trade_pct must be provided together.")
    if account_size_usd <= 0:
        raise ValueError("account_size_usd must be strictly positive.")
    if risk_per_trade_pct <= 0 or risk_per_trade_pct > 100:
        raise ValueError("risk_per_trade_pct must be in the (0, 100] interval.")

def _validate_backtest_inputs(
    stop_buffer_ticks: int,
    tick_size: float,
    max_leverage: float | None,
    account_size_usd: float | None,
    entry_delay_bars: int,
) -> None:
    """Validate non-risk backtest inputs."""
    if stop_buffer_ticks < 0:
        raise ValueError("stop_buffer_ticks must be non-negative.")
    if tick_size <= 0:
        raise ValueError("tick_size must be strictly positive.")
    if entry_delay_bars < 0:
        raise ValueError("entry_delay_bars must be non-negative.")
    if max_leverage is not None and max_leverage <= 0:
        raise ValueError("max_leverage must be strictly positive when provided.")
    if max_leverage is not None and account_size_usd is None:
        raise ValueError("account_size_usd must be provided when max_leverage is enabled.")

def _compute_risk_per_contract_usd(
    direction: int,
    entry_price: float,
    stop_price: float,
    execution_model: ExecutionModel,
    tick_value_usd: float,
) -> float:
    """Return per-contract risk in USD, including round-trip fees."""
    stop_exit_price = execution_model.apply_slippage(stop_price, direction, is_entry=False)
    stop_pnl_points = (stop_exit_price - entry_price) * direction
    stop_pnl_ticks = stop_pnl_points / execution_model.tick_size
    gross_loss_usd = abs(stop_pnl_ticks * tick_value_usd)
    fees_per_contract = execution_model.round_trip_fees(quantity=1)
    return gross_loss_usd + fees_per_contract

def _apply_leverage_cap(
    quantity: int,
    entry_price: float,
    account_size_usd: float | None,
    point_value_usd: float,
    max_leverage: float | None,
) -> int:
    """Cap quantity by a notional leverage constraint if configured."""
    if max_leverage is None or account_size_usd is None or point_value_usd <= 0:
        return quantity

    max_notional_usd = account_size_usd * max_leverage
    notional_per_contract_usd = entry_price * point_value_usd
    if notional_per_contract_usd <= 0:
        return 0

    leverage_capped_quantity = int(max_notional_usd / notional_per_contract_usd)
    return min(quantity, leverage_capped_quantity)

def run_backtest(
    df: pd.DataFrame,
    execution_model: ExecutionModel,
    tick_value_usd: float = DEFAULT_TICK_VALUE_USD,
    point_value_usd: float | None = None,
    time_exit: str = "15:55",
    stop_buffer_ticks: int = 0,
    target_multiple: float = 1.5,
    account_size_usd: float | None = None,
    risk_per_trade_pct: float | None = None,
    entry_on_next_open: bool = True,
    entry_delay_bars: int = 0,
    max_leverage: float | None = None,
) -> pd.DataFrame:
    """Run a straightforward signal-driven single-position backtest."""
    _validate_risk_inputs(account_size_usd, risk_per_trade_pct)
    _validate_backtest_inputs(
        stop_buffer_ticks=stop_buffer_ticks,
        tick_size=execution_model.tick_size,
        max_leverage=max_leverage,
        account_size_usd=account_size_usd,
        entry_delay_bars=entry_delay_bars,
    )

    trades = []
    trade_id = 1
    force_exit_time = dt.time.fromisoformat(time_exit)
    use_risk_sizing = account_size_usd is not None and risk_per_trade_pct is not None
    stop_buffer_points = stop_buffer_ticks * execution_model.tick_size
    point_value = point_value_usd if point_value_usd is not None else tick_value_usd / execution_model.tick_size

    for session_date, session_df in df.groupby("session_date", sort=True):
        session_df = session_df.sort_values("timestamp").reset_index(drop=True)
        signal_indices = session_df.index[session_df["signal"].fillna(0).ne(0)].tolist()
        if not signal_indices:
            continue

        next_search_index = 0

        for signal_idx in signal_indices:
            if signal_idx < next_search_index:
                continue

            row = session_df.loc[signal_idx]
            direction = int(row["signal"])
            or_high = row.get("or_high")
            or_low = row.get("or_low")
            if pd.isna(or_high) or pd.isna(or_low):
                continue

            base_entry_idx = signal_idx + 1 if entry_on_next_open else signal_idx
            entry_idx = base_entry_idx + entry_delay_bars
            if entry_idx >= len(session_df):
                continue

            entry_row = session_df.loc[entry_idx]
            delayed_entry = entry_delay_bars > 0
            use_open_entry = entry_on_next_open or delayed_entry
            raw_entry = entry_row["open"] if use_open_entry else row["close"]
            entry_price = execution_model.apply_slippage(raw_entry, direction, is_entry=True)
            entry_time = entry_row["timestamp"] if use_open_entry else row["timestamp"]

            if direction == 1:
                stop_price = float(or_low) - stop_buffer_points
            else:
                stop_price = float(or_high) + stop_buffer_points

            risk_points = (entry_price - stop_price) * direction
            if risk_points <= 0:
                continue

            target_price = entry_price + direction * target_multiple * risk_points
            risk_per_contract_usd = _compute_risk_per_contract_usd(
                direction=direction,
                entry_price=entry_price,
                stop_price=stop_price,
                execution_model=execution_model,
                tick_value_usd=tick_value_usd,
            )
            if risk_per_contract_usd <= 0:
                continue

            quantity = 1
            risk_budget_usd = None
            if use_risk_sizing:
                risk_budget_usd = account_size_usd * (risk_per_trade_pct / 100.0)
                quantity = int(risk_budget_usd / risk_per_contract_usd)
                if quantity < 1:
                    continue

            quantity = _apply_leverage_cap(
                quantity=quantity,
                entry_price=entry_price,
                account_size_usd=account_size_usd,
                point_value_usd=point_value,
                max_leverage=max_leverage,
            )
            if quantity < 1:
                continue

            trade_risk_usd = quantity * risk_per_contract_usd
            actual_risk_usd = trade_risk_usd if use_risk_sizing else None
            notional_usd = quantity * entry_price * point_value
            leverage_used = notional_usd / account_size_usd if account_size_usd is not None else None

            exit_scan_start = entry_idx if use_open_entry else entry_idx + 1
            if exit_scan_start >= len(session_df):
                continue

            exit_slice = session_df.loc[exit_scan_start:].copy()
            exit_reason = None
            raw_exit_price = None
            exit_idx = None

            if direction == 1:
                hit_mask = (exit_slice["low"] <= stop_price) | (exit_slice["high"] >= target_price)
            else:
                hit_mask = (exit_slice["high"] >= stop_price) | (exit_slice["low"] <= target_price)

            if hit_mask.any():
                exit_idx = hit_mask[hit_mask].index[0]
                exit_row = session_df.loc[exit_idx]
                if direction == 1:
                    if exit_row["low"] <= stop_price:
                        exit_reason = "stop"
                        raw_exit_price = stop_price
                    else:
                        exit_reason = "target"
                        raw_exit_price = target_price
                else:
                    if exit_row["high"] >= stop_price:
                        exit_reason = "stop"
                        raw_exit_price = stop_price
                    else:
                        exit_reason = "target"
                        raw_exit_price = target_price
            else:
                time_exit_mask = exit_slice["timestamp"].dt.time >= force_exit_time
                if time_exit_mask.any():
                    exit_idx = time_exit_mask[time_exit_mask].index[0]
                    exit_reason = "time_exit"
                else:
                    exit_idx = exit_slice.index[-1]
                    exit_reason = "eod_exit"
                raw_exit_price = float(session_df.loc[exit_idx, "close"])

            exit_row = session_df.loc[exit_idx]
            exit_price = execution_model.apply_slippage(raw_exit_price, direction, is_entry=False)
            pnl_points = (exit_price - entry_price) * direction
            pnl_ticks = pnl_points / execution_model.tick_size
            pnl_usd = pnl_ticks * tick_value_usd * quantity
            fees = execution_model.round_trip_fees(quantity=quantity)
            net_pnl = pnl_usd - fees

            trades.append(
                trade_to_record(
                    trade_id,
                    {
                        "session_date": session_date,
                        "direction": "long" if direction == 1 else "short",
                        "quantity": quantity,
                        "entry_time": entry_time,
                        "entry_price": entry_price,
                        "stop_price": stop_price,
                        "target_price": target_price,
                        "exit_time": exit_row["timestamp"],
                        "exit_price": exit_price,
                        "exit_reason": exit_reason,
                        "account_size_usd": account_size_usd if account_size_usd is not None else None,
                        "risk_per_trade_pct": risk_per_trade_pct if use_risk_sizing else None,
                        "risk_budget_usd": risk_budget_usd,
                        "risk_per_contract_usd": risk_per_contract_usd,
                        "actual_risk_usd": actual_risk_usd,
                        "trade_risk_usd": trade_risk_usd,
                        "notional_usd": notional_usd,
                        "leverage_used": leverage_used,
                        "pnl_points": pnl_points,
                        "pnl_ticks": pnl_ticks,
                        "pnl_usd": pnl_usd,
                        "fees": fees,
                        "net_pnl_usd": net_pnl,
                    },
                )
            )
            trade_id += 1
            next_search_index = int(exit_idx) + 1

    if not trades:
        return empty_trade_log()

    return pd.DataFrame(trades)

def _between_times(series: pd.Series, start: str, end: str) -> pd.Series:
    start_t = dt.time.fromisoformat(start)
    end_t = dt.time.fromisoformat(end)
    times = series.dt.time
    if start_t <= end_t:
        return (times >= start_t) & (times <= end_t)
    return (times >= start_t) | (times <= end_t)

def prepare_minute_dataset(
    dataset_path,
    baseline_entry: BaselineEntryConfig,
    atr_windows: tuple[int, ...] | list[int],
) -> pd.DataFrame:
    """Load and enrich the minute dataset once for all experiments."""
    raw = load_ohlcv_file(dataset_path)
    raw = clean_ohlcv(raw)
    feat = add_intraday_features(raw)
    feat = add_session_vwap(feat)
    feat = add_continuous_session_vwap(feat, session_start_hour=18)
    feat = compute_opening_range(feat, or_minutes=baseline_entry.or_minutes, opening_time=baseline_entry.opening_time)
    feat = add_atr(feat, window=sorted({int(x) for x in atr_windows if int(x) > 0}))
    feat = feat.sort_values("timestamp").reset_index(drop=True)
    return feat

def build_daily_reference(minute_df: pd.DataFrame) -> pd.DataFrame:
    """Build previous-day-aware daily features without look-ahead leakage."""
    rth_mask = _between_times(minute_df["timestamp"], "09:30:00", "16:00:00")
    rth = minute_df.loc[rth_mask, ["timestamp", "session_date", "open", "high", "low", "close"]].copy()

    def _first_open(group: pd.DataFrame) -> float:
        return float(group.iloc[0]["open"]) if not group.empty else np.nan

    daily = (
        rth.groupby("session_date", sort=True)
        .apply(
            lambda g: pd.Series(
                {
                    "open_rth": _first_open(g),
                    "high_rth": float(g["high"].max()),
                    "low_rth": float(g["low"].min()),
                    "close_rth": float(g.iloc[-1]["close"]),
                }
            )
        )
        .reset_index()
    )
    daily["range_rth"] = daily["high_rth"] - daily["low_rth"]
    daily["prev_close_rth"] = daily["close_rth"].shift(1)
    daily["prev_high_rth"] = daily["high_rth"].shift(1)
    daily["prev_low_rth"] = daily["low_rth"].shift(1)

    width = (daily["high_rth"] - daily["low_rth"]).replace(0.0, np.nan)
    close_pos = (daily["close_rth"] - daily["low_rth"]) / width
    daily["close_position"] = close_pos.fillna(0.5)

    nr4_raw = daily["range_rth"] <= daily["range_rth"].rolling(4, min_periods=4).min()
    nr7_raw = daily["range_rth"] <= daily["range_rth"].rolling(7, min_periods=7).min()
    inside_raw = (daily["high_rth"] < daily["prev_high_rth"]) & (daily["low_rth"] > daily["prev_low_rth"])
    outside_raw = (daily["high_rth"] > daily["prev_high_rth"]) & (daily["low_rth"] < daily["prev_low_rth"])

    tri_high = (daily["high_rth"] < daily["high_rth"].shift(1)) & (
        daily["high_rth"].shift(1) < daily["high_rth"].shift(2)
    )
    tri_low = (daily["low_rth"] > daily["low_rth"].shift(1)) & (daily["low_rth"].shift(1) > daily["low_rth"].shift(2))
    triangle_raw = tri_high & tri_low

    strong_close_raw = daily["close_position"] >= 0.75
    weak_close_raw = daily["close_position"] <= 0.25

    # Shift all pattern labels by one day to avoid leakage at day t open.
    daily["pattern_nr4"] = nr4_raw.shift(1).fillna(False)
    daily["pattern_nr7"] = nr7_raw.shift(1).fillna(False)
    daily["pattern_inside_day"] = inside_raw.shift(1).fillna(False)
    daily["pattern_outside_day"] = outside_raw.shift(1).fillna(False)
    daily["pattern_triangle"] = triangle_raw.shift(1).fillna(False)
    daily["pattern_strong_close"] = strong_close_raw.shift(1).fillna(False)
    daily["pattern_weak_close"] = weak_close_raw.shift(1).fillna(False)

    daily["pattern_nr4_or_nr7"] = daily["pattern_nr4"] | daily["pattern_nr7"]
    daily["pattern_nr4_or_triangle"] = daily["pattern_nr4"] | daily["pattern_triangle"]
    daily["pattern_nr4_or_nr7_or_triangle"] = (
        daily["pattern_nr4"] | daily["pattern_nr7"] | daily["pattern_triangle"]
    )

    return daily

def attach_daily_reference(minute_df: pd.DataFrame, daily_df: pd.DataFrame) -> pd.DataFrame:
    merged = minute_df.merge(daily_df, on="session_date", how="left")
    merged["open_prevclose_max"] = np.maximum(
        pd.to_numeric(merged["open_rth"], errors="coerce"),
        pd.to_numeric(merged["prev_close_rth"], errors="coerce"),
    )
    merged["open_prevclose_min"] = np.minimum(
        pd.to_numeric(merged["open_rth"], errors="coerce"),
        pd.to_numeric(merged["prev_close_rth"], errors="coerce"),
    )
    return merged

def build_candidate_universe(
    minute_df: pd.DataFrame,
    baseline_entry: BaselineEntryConfig,
) -> pd.DataFrame:
    """Generate all candidate breakout rows before one-trade/day reduction."""
    strategy = ORBStrategy(
        or_minutes=baseline_entry.or_minutes,
        direction=baseline_entry.direction,
        one_trade_per_day=False,
        entry_buffer_ticks=baseline_entry.entry_buffer_ticks,
        stop_buffer_ticks=baseline_entry.stop_buffer_ticks,
        target_multiple=baseline_entry.target_multiple,
        opening_time=baseline_entry.opening_time,
        time_exit=baseline_entry.time_exit,
        account_size_usd=baseline_entry.account_size_usd,
        risk_per_trade_pct=baseline_entry.risk_per_trade_pct,
        tick_size=baseline_entry.tick_size,
        vwap_confirmation=baseline_entry.vwap_confirmation,
        vwap_column=baseline_entry.vwap_column,
    )
    signals = strategy.generate_signals(minute_df)
    signals = signals.sort_values("timestamp").reset_index(drop=True)
    signals["candidate_base_pass"] = (
        signals["raw_signal"].eq(1)
        & signals["atr_filter_pass"].fillna(False)
        & signals["direction_filter_pass"].fillna(False)
    )
    return signals

def _schedule_mask(minute_of_day: pd.Series, schedule: str) -> pd.Series:
    if schedule == "continuous_on_bar_close":
        return pd.Series(True, index=minute_of_day.index)
    if schedule == "every_5m":
        return minute_of_day.mod(5).eq(0)
    if schedule == "every_15m":
        return minute_of_day.mod(15).eq(0)
    raise ValueError(f"Unsupported schedule: {schedule}")

def _consecutive_true_within_session(mask: pd.Series, session_dates: pd.Series) -> pd.Series:
    out = pd.Series(0, index=mask.index, dtype=int)
    for _, idx in session_dates.groupby(session_dates).groups.items():
        session_mask = mask.loc[idx].fillna(False)
        counter = 0
        values = []
        for flag in session_mask.tolist():
            counter = counter + 1 if flag else 0
            values.append(counter)
        out.loc[idx] = values
    return out

def compute_noise_sigma(minute_df: pd.DataFrame, lookback: int) -> pd.Series:
    """Compute noise sigma(t,m) as rolling mean abs move-from-open over previous sessions."""
    rth_mask = _between_times(minute_df["timestamp"], "09:30:00", "16:00:00")
    base = minute_df.loc[rth_mask, ["session_date", "minute_of_day", "close", "open_rth"]].copy()
    base = base.dropna(subset=["open_rth"])  # only sessions with known RTH open
    base["abs_move_from_open"] = (base["close"] / base["open_rth"] - 1.0).abs()

    pivot = base.pivot_table(index="session_date", columns="minute_of_day", values="abs_move_from_open", aggfunc="last")
    min_periods = max(2, int(round(lookback * 0.6)))
    min_periods = min(int(lookback), int(min_periods))
    sigma = pivot.shift(1).rolling(lookback, min_periods=min_periods).mean()

    sigma_long = (
        sigma.stack(dropna=False)
        .rename("noise_sigma")
        .reset_index()
        .rename(columns={"level_1": "minute_of_day"})
    )

    merged = minute_df[["session_date", "minute_of_day"]].merge(
        sigma_long,
        on=["session_date", "minute_of_day"],
        how="left",
    )
    return pd.to_numeric(merged["noise_sigma"], errors="coerce")

def dynamic_gate_mask(
    candidate_df: pd.DataFrame,
    config: DynamicThresholdConfig,
    noise_sigma: pd.Series | None,
    atr_col: str,
) -> pd.Series:
    """Return per-row dynamic-threshold pass mask (True when disabled)."""
    if config.mode == "disabled":
        return pd.Series(True, index=candidate_df.index)

    close = pd.to_numeric(candidate_df["close"], errors="coerce")
    or_high = pd.to_numeric(candidate_df["or_high"], errors="coerce")
    threshold = pd.Series(np.nan, index=candidate_df.index, dtype=float)

    if config.mode in {"noise_area_gate", "noise_area_gate_plus_close_confirmation", "noise_area_gate_plus_discrete_schedule"}:
        if noise_sigma is None:
            return pd.Series(False, index=candidate_df.index)

        sigma = pd.to_numeric(noise_sigma, errors="coerce").reindex(candidate_df.index)
        base_ref = pd.to_numeric(candidate_df["open_prevclose_max"], errors="coerce")

        upper_noise = base_ref * (1.0 + float(config.noise_vm) * sigma)
        noise_abs = (base_ref * sigma).abs()
        if config.threshold_style == "max_or_high_noise":
            threshold = np.maximum(or_high, upper_noise)
        elif config.threshold_style == "or_high_plus_k_noise_abs":
            threshold = or_high + float(config.noise_k) * noise_abs
        else:
            raise ValueError(f"Unsupported threshold_style: {config.threshold_style}")

    elif config.mode == "atr_threshold_gate":
        atr_value = pd.to_numeric(candidate_df.get(atr_col, np.nan), errors="coerce")
        threshold = or_high + float(config.atr_k) * atr_value

    elif config.mode == "close_confirmation_gate":
        threshold = or_high

    else:
        raise ValueError(f"Unsupported dynamic mode: {config.mode}")

    above = close > threshold
    confirm_bars = max(1, int(config.confirm_bars))
    streak = _consecutive_true_within_session(above.fillna(False), candidate_df["session_date"])
    confirm_ok = streak >= confirm_bars

    schedule = config.schedule
    if config.mode == "close_confirmation_gate":
        schedule = "continuous_on_bar_close"
    if config.mode == "noise_area_gate":
        schedule = "continuous_on_bar_close"

    schedule_ok = _schedule_mask(candidate_df["minute_of_day"], schedule)
    return above.fillna(False) & confirm_ok & schedule_ok

def compression_mask(candidate_df: pd.DataFrame, config: CompressionConfig) -> pd.Series:
    """Return session-level compression pattern mask mapped to candidate rows."""
    mode = config.mode
    if mode == "none":
        return pd.Series(True, index=candidate_df.index)

    column_map = {
        "nr4": "pattern_nr4",
        "nr7": "pattern_nr7",
        "triangle": "pattern_triangle",
        "nr4_or_nr7": "pattern_nr4_or_nr7",
        "nr4_or_triangle": "pattern_nr4_or_triangle",
        "nr4_or_nr7_or_triangle": "pattern_nr4_or_nr7_or_triangle",
        "inside_day": "pattern_inside_day",
        "outside_day": "pattern_outside_day",
        "strong_close": "pattern_strong_close",
        "weak_close": "pattern_weak_close",
    }
    col = column_map.get(mode)
    if col is None or col not in candidate_df.columns:
        raise ValueError(f"Unsupported compression mode: {mode}")
    return candidate_df[col].fillna(False).astype(bool)

def first_pass_signal_rows(
    candidate_df: pd.DataFrame,
    pass_mask: pd.Series,
) -> pd.DataFrame:
    """Select first passing candidate row per session (one trade max/day)."""
    working = candidate_df.loc[pass_mask.fillna(False)].copy()
    if working.empty:
        return working
    working = working.sort_values("timestamp")
    return working.groupby("session_date", sort=True).head(1).copy()

def calibrate_ensemble_thresholds(
    selected_signal_rows: pd.DataFrame,
    is_sessions: list,
    atr_col: str,
    q_lows_pct: tuple[int, ...],
    q_highs_pct: tuple[int, ...],
) -> list[tuple[float, float]]:
    """Calibrate ATR quantile bands on IS only."""
    is_values = pd.to_numeric(
        selected_signal_rows.loc[selected_signal_rows["session_date"].isin(set(is_sessions)), atr_col],
        errors="coerce",
    ).dropna()
    thresholds: list[tuple[float, float]] = []
    for q_low in q_lows_pct:
        for q_high in q_highs_pct:
            if q_low >= q_high:
                continue
            low = float(is_values.quantile(float(q_low) / 100.0)) if not is_values.empty else np.nan
            high = float(is_values.quantile(float(q_high) / 100.0)) if not is_values.empty else np.nan
            if math.isfinite(low) and math.isfinite(high) and low < high:
                thresholds.append((low, high))
    return thresholds

def apply_ensemble_selection(
    selected_signal_rows: pd.DataFrame,
    atr_col: str,
    thresholds: list[tuple[float, float]],
    vote_threshold: float,
    compression_config: CompressionConfig,
) -> pd.DataFrame:
    """Apply ATR-ensemble vote and optional compression soft-vote bonus."""
    out = selected_signal_rows.copy()
    if out.empty:
        out["ensemble_score"] = pd.Series(dtype=float)
        out["ensemble_selected"] = pd.Series(dtype=bool)
        return out

    atr = pd.to_numeric(out[atr_col], errors="coerce")
    if not thresholds:
        out["ensemble_score"] = 0.0
        out["ensemble_selected"] = False
        return out

    pass_columns: list[pd.Series] = []
    for low, high in thresholds:
        pass_columns.append(atr.between(low, high, inclusive="both"))

    pass_frame = pd.concat(pass_columns, axis=1)
    pass_count = pass_frame.sum(axis=1).astype(float)
    n_votes = float(pass_frame.shape[1])

    if compression_config.usage == "soft_vote_bonus" and compression_config.mode != "none":
        comp = compression_mask(out, compression_config).astype(float)
        bonus = float(max(0.0, compression_config.soft_bonus_votes))
        score = (pass_count + bonus * comp) / (n_votes + bonus if n_votes + bonus > 0 else 1.0)
    else:
        score = pass_count / n_votes

    out["ensemble_score"] = score
    out["ensemble_selected"] = out["ensemble_score"] >= float(vote_threshold)
    return out

def build_signal_frame_for_backtest(
    minute_df: pd.DataFrame,
    selected_signal_rows: pd.DataFrame,
) -> pd.DataFrame:
    """Return a backtest-ready dataframe with sparse signal column."""
    if selected_signal_rows.empty:
        out = minute_df.copy()
        out["signal"] = 0
        return out.iloc[:0].copy()

    selected_indices = pd.Index(selected_signal_rows.index)
    selected_sessions = set(pd.to_datetime(selected_signal_rows["session_date"]).dt.date)
    out = minute_df.loc[pd.to_datetime(minute_df["session_date"]).dt.date.isin(selected_sessions)].copy()
    out = out.sort_values("timestamp").reset_index(drop=True)
    out["signal"] = 0

    # Preserve selection by timestamp + session_date after reindexing.
    key = selected_signal_rows[["session_date", "timestamp"]].copy()
    key["_selected"] = 1
    out = out.merge(key, on=["session_date", "timestamp"], how="left")
    out.loc[out["_selected"].eq(1), "signal"] = 1
    out = out.drop(columns=["_selected"])
    return out

def _compute_risk_per_contract_usd(
    direction: int,
    entry_price: float,
    stop_price: float,
    execution_model: ExecutionModel,
    tick_value_usd: float,
) -> float:
    stop_exit = execution_model.apply_slippage(stop_price, direction, is_entry=False)
    pnl_points = (stop_exit - entry_price) * direction
    pnl_ticks = pnl_points / execution_model.tick_size
    gross_loss = abs(pnl_ticks * tick_value_usd)
    return gross_loss + execution_model.round_trip_fees(quantity=1)

def _apply_leverage_cap(
    quantity: int,
    entry_price: float,
    account_size_usd: float,
    point_value_usd: float,
    max_leverage: float | None,
) -> int:
    if max_leverage is None:
        return quantity
    max_notional = account_size_usd * max_leverage
    per_contract_notional = entry_price * point_value_usd
    if per_contract_notional <= 0:
        return 0
    cap_qty = int(max_notional / per_contract_notional)
    return min(quantity, cap_qty)

def _resolve_force_exit_time(exit_cfg: ExitConfig, baseline: BaselineEntryConfig) -> dt.time:
    force_time = exit_cfg.force_exit_time or baseline.time_exit
    return dt.time.fromisoformat(force_time)

def _stagnation_hit(
    bars_since_entry: int,
    stagnation_bars: int | None,
    max_favorable_r: float,
    min_r_required: float,
) -> bool:
    if stagnation_bars is None:
        return False
    return bars_since_entry >= int(stagnation_bars) and max_favorable_r < float(min_r_required)

def run_exit_variant_backtest(
    signal_df: pd.DataFrame,
    execution_model: ExecutionModel,
    baseline: BaselineEntryConfig,
    exit_cfg: ExitConfig,
    tick_value_usd: float = DEFAULT_TICK_VALUE_USD,
    point_value_usd: float | None = None,
    max_leverage: float | None = None,
) -> pd.DataFrame:
    """Run baseline or overlay exit logic while keeping entry logic unchanged."""
    mode = exit_cfg.mode
    if mode == "baseline":
        return run_backtest(
            signal_df,
            execution_model=execution_model,
            tick_value_usd=tick_value_usd,
            point_value_usd=point_value_usd,
            time_exit=baseline.time_exit,
            stop_buffer_ticks=baseline.stop_buffer_ticks,
            target_multiple=baseline.target_multiple,
            account_size_usd=baseline.account_size_usd,
            risk_per_trade_pct=baseline.risk_per_trade_pct,
            entry_on_next_open=baseline.entry_on_next_open,
            max_leverage=max_leverage,
        )

    if signal_df.empty:
        return empty_trade_log()

    point_value = point_value_usd if point_value_usd is not None else tick_value_usd / execution_model.tick_size
    stop_buffer_points = baseline.stop_buffer_ticks * execution_model.tick_size
    force_exit_time = _resolve_force_exit_time(exit_cfg, baseline)

    trades: list[dict[str, object]] = []
    trade_id = 1

    required_vwap_modes = {
        "fail_fast_vwap",
        "trailing_vwap",
        "trailing_struct_plus_vwap",
        "partial_1R_then_vwap",
        "time_stop_plus_vwap",
    }

    for session_date, session_df in signal_df.groupby("session_date", sort=True):
        frame = session_df.sort_values("timestamp").reset_index(drop=True)
        signal_indices = frame.index[frame["signal"].fillna(0).ne(0)].tolist()
        if not signal_indices:
            continue

        next_search_index = 0
        for signal_idx in signal_indices:
            if signal_idx < next_search_index:
                continue

            signal_row = frame.loc[signal_idx]
            direction = int(signal_row["signal"])
            if direction != 1:
                continue

            or_high = pd.to_numeric(signal_row.get("or_high"), errors="coerce")
            or_low = pd.to_numeric(signal_row.get("or_low"), errors="coerce")
            if pd.isna(or_high) or pd.isna(or_low):
                continue

            entry_idx = signal_idx + 1 if baseline.entry_on_next_open else signal_idx
            if entry_idx >= len(frame):
                continue

            entry_row = frame.loc[entry_idx]
            raw_entry = float(entry_row["open"] if baseline.entry_on_next_open else signal_row["close"])
            entry_price = execution_model.apply_slippage(raw_entry, direction, is_entry=True)
            entry_time = entry_row["timestamp"] if baseline.entry_on_next_open else signal_row["timestamp"]

            initial_stop = float(or_low) - stop_buffer_points
            risk_points = (entry_price - initial_stop) * direction
            if risk_points <= 0:
                continue

            full_target = entry_price + direction * baseline.target_multiple * risk_points
            partial_target = entry_price + direction * risk_points

            risk_per_contract = _compute_risk_per_contract_usd(
                direction=direction,
                entry_price=entry_price,
                stop_price=initial_stop,
                execution_model=execution_model,
                tick_value_usd=tick_value_usd,
            )
            if risk_per_contract <= 0:
                continue

            risk_budget = baseline.account_size_usd * (baseline.risk_per_trade_pct / 100.0)
            quantity = int(risk_budget / risk_per_contract)
            if quantity < 1:
                continue

            quantity = _apply_leverage_cap(
                quantity=quantity,
                entry_price=entry_price,
                account_size_usd=baseline.account_size_usd,
                point_value_usd=point_value,
                max_leverage=max_leverage,
            )
            if quantity < 1:
                continue

            risk_usd = quantity * risk_per_contract
            notional_usd = quantity * entry_price * point_value
            leverage_used = notional_usd / baseline.account_size_usd if baseline.account_size_usd > 0 else np.nan

            exit_scan_start = entry_idx if baseline.entry_on_next_open else entry_idx + 1
            if exit_scan_start >= len(frame):
                continue

            current_stop = float(initial_stop)
            structural_stop = float(initial_stop)

            remaining_qty = int(quantity)
            partial_qty = int(math.floor(quantity * float(exit_cfg.partial_fraction)))
            partial_qty = max(0, min(partial_qty, remaining_qty - 1))
            runner_qty = remaining_qty

            legs: list[dict[str, object]] = []
            max_favorable_r = 0.0
            bars_since_entry = 0
            exit_idx = None
            final_reason = None

            for bar_idx in range(exit_scan_start, len(frame)):
                bar = frame.loc[bar_idx]
                bars_since_entry += 1

                high = float(bar["high"])
                low = float(bar["low"])
                close = float(bar["close"])
                ts = bar["timestamp"]
                ts_time = ts.time()
                vwap = pd.to_numeric(bar.get("continuous_session_vwap", np.nan), errors="coerce")
                if pd.isna(vwap):
                    vwap = pd.to_numeric(bar.get("session_vwap", np.nan), errors="coerce")

                favorable_r = max(0.0, (high - entry_price) / risk_points)
                max_favorable_r = max(max_favorable_r, favorable_r)

                stop_hit = low <= current_stop
                target_hit = high >= full_target

                if mode in {"trailing_vwap", "trailing_struct_plus_vwap", "partial_1R_then_vwap", "time_stop_plus_vwap"}:
                    # trailing-based modes keep target only for staged leg opening, not for full one-shot profit taking.
                    target_hit = False

                if stop_hit:
                    legs.append(
                        {
                            "qty": runner_qty,
                            "raw_exit_price": current_stop,
                            "exit_time": ts,
                            "reason": "stop",
                        }
                    )
                    final_reason = "stop"
                    exit_idx = bar_idx
                    runner_qty = 0
                    break

                if mode == "partial_1R_then_vwap" and partial_qty > 0 and runner_qty == quantity and high >= partial_target:
                    legs.append(
                        {
                            "qty": partial_qty,
                            "raw_exit_price": partial_target,
                            "exit_time": ts,
                            "reason": "partial_1R",
                        }
                    )
                    runner_qty -= partial_qty

                if target_hit and runner_qty > 0:
                    legs.append(
                        {
                            "qty": runner_qty,
                            "raw_exit_price": full_target,
                            "exit_time": ts,
                            "reason": "target",
                        }
                    )
                    final_reason = "target"
                    exit_idx = bar_idx
                    runner_qty = 0
                    break

                if mode == "fail_fast_vwap" and pd.notna(vwap) and close < float(vwap) and runner_qty > 0:
                    legs.append(
                        {
                            "qty": runner_qty,
                            "raw_exit_price": close,
                            "exit_time": ts,
                            "reason": "fail_fast_vwap",
                        }
                    )
                    final_reason = "fail_fast_vwap"
                    exit_idx = bar_idx
                    runner_qty = 0
                    break

                if mode == "time_stop_plus_vwap" and _stagnation_hit(
                    bars_since_entry=bars_since_entry,
                    stagnation_bars=exit_cfg.stagnation_bars,
                    max_favorable_r=max_favorable_r,
                    min_r_required=exit_cfg.stagnation_min_r_multiple,
                ) and runner_qty > 0:
                    legs.append(
                        {
                            "qty": runner_qty,
                            "raw_exit_price": close,
                            "exit_time": ts,
                            "reason": "stagnation_time_stop",
                        }
                    )
                    final_reason = "stagnation_time_stop"
                    exit_idx = bar_idx
                    runner_qty = 0
                    break

                if ts_time >= force_exit_time and runner_qty > 0:
                    legs.append(
                        {
                            "qty": runner_qty,
                            "raw_exit_price": close,
                            "exit_time": ts,
                            "reason": "time_exit",
                        }
                    )
                    final_reason = "time_exit"
                    exit_idx = bar_idx
                    runner_qty = 0
                    break

                if mode in required_vwap_modes and pd.notna(vwap):
                    vwap_val = float(vwap)
                    if mode in {"trailing_struct_plus_vwap", "partial_1R_then_vwap", "time_stop_plus_vwap"}:
                        structural_stop = max(structural_stop, low - stop_buffer_points)
                        current_stop = max(current_stop, structural_stop, vwap_val)
                    else:
                        current_stop = max(current_stop, vwap_val)

            if runner_qty > 0:
                last_row = frame.iloc[-1]
                legs.append(
                    {
                        "qty": runner_qty,
                        "raw_exit_price": float(last_row["close"]),
                        "exit_time": last_row["timestamp"],
                        "reason": "eod_exit",
                    }
                )
                final_reason = "eod_exit"
                exit_idx = len(frame) - 1

            gross_pnl_usd = 0.0
            final_exit_price = np.nan
            final_exit_time = None
            for leg in legs:
                qty = int(leg["qty"])
                if qty <= 0:
                    continue
                exit_price = execution_model.apply_slippage(float(leg["raw_exit_price"]), direction, is_entry=False)
                pnl_points = (exit_price - entry_price) * direction
                pnl_ticks = pnl_points / execution_model.tick_size
                gross_pnl_usd += pnl_ticks * tick_value_usd * qty
                final_exit_price = exit_price
                final_exit_time = leg["exit_time"]

            fees = execution_model.round_trip_fees(quantity=quantity)
            net_pnl = gross_pnl_usd - fees
            pnl_points_total = net_pnl / (tick_value_usd * quantity / execution_model.tick_size) if quantity > 0 else 0.0
            pnl_ticks_total = pnl_points_total / execution_model.tick_size

            trades.append(
                trade_to_record(
                    trade_id,
                    {
                        "session_date": session_date,
                        "direction": "long",
                        "quantity": quantity,
                        "entry_time": entry_time,
                        "entry_price": entry_price,
                        "stop_price": initial_stop,
                        "target_price": full_target,
                        "exit_time": final_exit_time,
                        "exit_price": final_exit_price,
                        "exit_reason": final_reason,
                        "account_size_usd": baseline.account_size_usd,
                        "risk_per_trade_pct": baseline.risk_per_trade_pct,
                        "risk_budget_usd": risk_budget,
                        "risk_per_contract_usd": risk_per_contract,
                        "actual_risk_usd": risk_usd,
                        "trade_risk_usd": risk_usd,
                        "notional_usd": notional_usd,
                        "leverage_used": leverage_used,
                        "pnl_points": pnl_points_total,
                        "pnl_ticks": pnl_ticks_total,
                        "pnl_usd": gross_pnl_usd,
                        "fees": fees,
                        "net_pnl_usd": net_pnl,
                    },
                )
            )
            trade_id += 1
            next_search_index = int(exit_idx) + 1 if exit_idx is not None else next_search_index

    if not trades:
        return empty_trade_log()
    return pd.DataFrame(trades)

def _daily_pnl_from_trades(trades: pd.DataFrame, sessions: list) -> pd.Series:
    idx = pd.Index(pd.to_datetime(pd.Index(sessions)).date)
    if idx.empty:
        return pd.Series(dtype=float)
    if trades.empty:
        return pd.Series(0.0, index=idx, dtype=float)

    grouped = trades.groupby(pd.to_datetime(trades["session_date"]).dt.date)["net_pnl_usd"].sum()
    return grouped.reindex(idx, fill_value=0.0).astype(float)

def _run_lengths(mask: pd.Series) -> list[int]:
    lengths: list[int] = []
    current = 0
    for flag in mask.astype(bool).tolist():
        if flag:
            current += 1
        elif current > 0:
            lengths.append(current)
            current = 0
    if current > 0:
        lengths.append(current)
    return lengths

def _challenge_outcome(
    pnl_path: np.ndarray,
    target_usd: float,
    max_dd_usd: float,
) -> tuple[str, int]:
    cum = 0.0
    for i, value in enumerate(pnl_path, start=1):
        cum += float(value)
        if cum >= target_usd:
            return "success", i
        if cum <= -max_dd_usd:
            return "fail", i
    return "open", len(pnl_path)

def simulate_prop_challenge(
    daily_pnl: pd.Series,
    initial_capital: float,
    target_return_pct: float = 0.06,
    max_drawdown_pct: float = 0.04,
    bootstrap_paths: int = 3000,
    random_seed: int = 42,
) -> dict[str, float]:
    series = pd.to_numeric(daily_pnl, errors="coerce").fillna(0.0)
    if series.empty:
        return {
            "challenge_target_return_pct": target_return_pct,
            "challenge_drawdown_limit_pct": max_drawdown_pct,
            "challenge_target_usd": initial_capital * target_return_pct,
            "challenge_drawdown_limit_usd": initial_capital * max_drawdown_pct,
            "challenge_hit_probability": 0.0,
            "challenge_hit_probability_bootstrap": 0.0,
            "challenge_hit_probability_rolling": 0.0,
            "challenge_fail_fast_probability": 0.0,
            "challenge_median_days_to_target": np.nan,
            "challenge_median_days_to_target_bootstrap": np.nan,
            "challenge_median_days_to_target_rolling": np.nan,
            "challenge_success_paths_bootstrap": 0.0,
            "challenge_success_paths_rolling": 0.0,
        }

    target_usd = float(initial_capital * target_return_pct)
    max_dd_usd = float(initial_capital * max_drawdown_pct)
    values = series.to_numpy(dtype=float)

    rolling_days: list[int] = []
    rolling_success = 0
    rolling_fail_fast = 0
    for start in range(len(values)):
        outcome, days = _challenge_outcome(values[start:], target_usd=target_usd, max_dd_usd=max_dd_usd)
        if outcome == "success":
            rolling_success += 1
            rolling_days.append(days)
        if outcome == "fail" and days <= 5:
            rolling_fail_fast += 1

    rolling_total = max(1, len(values))
    rolling_rate = rolling_success / rolling_total
    fail_fast_rate = rolling_fail_fast / rolling_total

    rng = np.random.default_rng(random_seed)
    bootstrap_days: list[int] = []
    bootstrap_success = 0
    horizon = len(values)
    for _ in range(max(0, int(bootstrap_paths))):
        sampled = rng.choice(values, size=horizon, replace=True)
        outcome, days = _challenge_outcome(sampled, target_usd=target_usd, max_dd_usd=max_dd_usd)
        if outcome == "success":
            bootstrap_success += 1
            bootstrap_days.append(days)

    bootstrap_rate = bootstrap_success / bootstrap_paths if bootstrap_paths > 0 else 0.0
    combined_rate = 0.5 * rolling_rate + 0.5 * bootstrap_rate

    return {
        "challenge_target_return_pct": target_return_pct,
        "challenge_drawdown_limit_pct": max_drawdown_pct,
        "challenge_target_usd": target_usd,
        "challenge_drawdown_limit_usd": max_dd_usd,
        "challenge_hit_probability": float(combined_rate),
        "challenge_hit_probability_bootstrap": float(bootstrap_rate),
        "challenge_hit_probability_rolling": float(rolling_rate),
        "challenge_fail_fast_probability": float(fail_fast_rate),
        "challenge_median_days_to_target": float(np.median(bootstrap_days)) if bootstrap_days else np.nan,
        "challenge_median_days_to_target_bootstrap": float(np.median(bootstrap_days)) if bootstrap_days else np.nan,
        "challenge_median_days_to_target_rolling": float(np.median(rolling_days)) if rolling_days else np.nan,
        "challenge_success_paths_bootstrap": float(bootstrap_success),
        "challenge_success_paths_rolling": float(rolling_success),
    }

def compute_extended_metrics(
    trades: pd.DataFrame,
    signal_df: pd.DataFrame | None,
    sessions: list,
    initial_capital: float,
    bootstrap_paths: int,
    random_seed: int,
) -> dict[str, float | int | str | bool]:
    base = compute_metrics(
        trades,
        signal_df=signal_df,
        session_dates=sessions,
        initial_capital=initial_capital,
    )

    daily = _daily_pnl_from_trades(trades, sessions)
    cumulative_daily = daily.cumsum()
    equity = initial_capital + cumulative_daily
    peak = equity.cummax()
    drawdown = equity - peak

    if len(daily) >= 3:
        worst_3day_run = float(daily.rolling(3).sum().min())
    else:
        worst_3day_run = float(daily.sum()) if not daily.empty else 0.0

    if len(daily) >= 5:
        worst_5day_drawdown = float(daily.rolling(5).sum().min())
    else:
        worst_5day_drawdown = float(daily.sum()) if not daily.empty else 0.0

    avg_daily_pnl = float(daily.mean()) if not daily.empty else 0.0
    worst_day = float(daily.min()) if not daily.empty else 0.0

    losing_streaks = _run_lengths((daily < 0).astype(bool))
    max_losing_streak_days = max(losing_streaks, default=0)

    if not trades.empty:
        hold_minutes = (
            (pd.to_datetime(trades["exit_time"]) - pd.to_datetime(trades["entry_time"]))
            .dt.total_seconds()
            .div(60.0)
        )
        avg_hold_minutes = float(hold_minutes.mean())
    else:
        avg_hold_minutes = 0.0

    return_over_drawdown = float(base.get("cumulative_pnl", 0.0)) / max(abs(float(base.get("max_drawdown", 0.0))), 1.0)

    underwater = drawdown < 0
    underwater_pct = float(underwater.mean()) if len(underwater) > 0 else 0.0
    ulcer_index = float(np.sqrt(np.mean(np.square((drawdown / initial_capital) * 100.0)))) if len(drawdown) > 0 else 0.0

    challenge = simulate_prop_challenge(
        daily_pnl=daily,
        initial_capital=initial_capital,
        target_return_pct=0.06,
        max_drawdown_pct=0.04,
        bootstrap_paths=bootstrap_paths,
        random_seed=random_seed,
    )

    out: dict[str, float | int | str | bool] = dict(base)
    out.update(
        {
            "net_pnl": float(base.get("cumulative_pnl", 0.0)),
            "return_over_drawdown": return_over_drawdown,
            "avg_daily_pnl": avg_daily_pnl,
            "worst_day": worst_day,
            "worst_3day_run": worst_3day_run,
            "worst_5day_drawdown": worst_5day_drawdown,
            "max_losing_streak": int(base.get("longest_loss_streak", 0)),
            "max_losing_streak_days": int(max_losing_streak_days),
            "nb_trades": int(base.get("n_trades", 0)),
            "pct_days_traded": float(base.get("percent_of_days_traded", 0.0)),
            "avg_time_in_position_min": avg_hold_minutes,
            "hit_ratio": float(base.get("win_rate", 0.0)),
            "avg_winner": float(base.get("avg_win", 0.0)),
            "avg_loser": float(base.get("avg_loss", 0.0)),
            "time_underwater_pct": underwater_pct,
            "ulcer_index": ulcer_index,
        }
    )
    out.update(challenge)
    return out

def compute_scores(metrics_row: pd.Series) -> dict[str, float]:
    """Compute academic and prop-oriented ranking scores."""
    sharpe = float(metrics_row.get("sharpe_ratio", 0.0))
    pf = float(metrics_row.get("profit_factor", 0.0))
    expectancy = float(metrics_row.get("expectancy", 0.0))
    ret_dd = float(metrics_row.get("return_over_drawdown", 0.0))

    max_dd = abs(float(metrics_row.get("max_drawdown", 0.0)))
    worst_5d = abs(float(metrics_row.get("worst_5day_drawdown", 0.0)))
    losing_streak = float(metrics_row.get("max_losing_streak", 0.0))
    fail_fast = float(metrics_row.get("challenge_fail_fast_probability", 0.0))
    challenge_prob = float(metrics_row.get("challenge_hit_probability", 0.0))
    median_days = pd.to_numeric(pd.Series([metrics_row.get("challenge_median_days_to_target")]), errors="coerce").iloc[0]
    pnl_oos = float(metrics_row.get("net_pnl", 0.0))

    pf_capped = pf if math.isfinite(pf) else 3.0

    academic_score = (
        0.45 * sharpe
        + 0.25 * (pf_capped - 1.0)
        + 0.20 * np.tanh(expectancy / 50.0)
        + 0.10 * np.tanh(ret_dd / 4.0)
    )

    speed_bonus = 0.0
    if pd.notna(median_days) and float(median_days) > 0:
        speed_bonus = float(np.clip(40.0 / float(median_days), 0.0, 2.0))

    prop_score = (
        2.2 * np.tanh(ret_dd / 3.0)
        + 2.4 * challenge_prob
        + 0.6 * np.tanh((pf_capped - 1.0) / 0.4)
        + 0.5 * np.tanh(expectancy / 40.0)
        + 0.3 * np.tanh(pnl_oos / 4000.0)
        + 0.7 * speed_bonus
        - 1.8 * np.tanh(max_dd / 2500.0)
        - 1.4 * np.tanh(worst_5d / 1800.0)
        - 0.9 * np.tanh(max(losing_streak - 2.0, 0.0) / 5.0)
        - 1.2 * fail_fast
    )

    return {
        "academic_score": float(academic_score),
        "prop_score": float(prop_score),
    }

def _split_sessions(all_sessions: list, is_fraction: float) -> tuple[list, list]:
    if len(all_sessions) < 2:
        raise ValueError("Not enough sessions for IS/OOS split.")
    split_idx = int(len(all_sessions) * float(is_fraction))
    split_idx = max(1, min(len(all_sessions) - 1, split_idx))
    return all_sessions[:split_idx], all_sessions[split_idx:]

def _serialize_config(config: ExperimentConfig) -> str:
    return json.dumps(
        {
            "name": config.name,
            "stage": config.stage,
            "family": config.family,
            "baseline_entry": asdict(config.baseline_entry),
            "baseline_ensemble": asdict(config.baseline_ensemble),
            "compression": asdict(config.compression),
            "exit": asdict(config.exit),
            "dynamic_threshold": asdict(config.dynamic_threshold),
        },
        sort_keys=True,
    )

def _experiment_from_json(payload: str) -> ExperimentConfig:
    data = json.loads(payload)
    return ExperimentConfig(
        name=data["name"],
        stage=data["stage"],
        family=data["family"],
        baseline_entry=BaselineEntryConfig(**data["baseline_entry"]),
        baseline_ensemble=BaselineEnsembleConfig(**data["baseline_ensemble"]),
        compression=CompressionConfig(**data["compression"]),
        exit=ExitConfig(**data["exit"]),
        dynamic_threshold=DynamicThresholdConfig(**data["dynamic_threshold"]),
    )

def _evaluate_experiment(
    experiment: ExperimentConfig,
    context: CampaignContext,
    bootstrap_paths: int,
    random_seed: int,
    keep_details: bool = False,
    max_leverage: float | None = None,
) -> tuple[dict[str, object], dict[str, object] | None]:
    candidate_df = context.candidate_base_df
    ensemble = experiment.baseline_ensemble
    atr_col = f"atr_{ensemble.atr_window}"

    if atr_col not in candidate_df.columns:
        row = {
            "name": experiment.name,
            "stage": experiment.stage,
            "family": experiment.family,
            "config_json": _serialize_config(experiment),
            "status": "missing_atr_column",
        }
        return row, None

    comp_mask = compression_mask(candidate_df, experiment.compression)
    noise_sigma = None
    dyn_cfg = experiment.dynamic_threshold
    if dyn_cfg.mode in {
        "noise_area_gate",
        "noise_area_gate_plus_close_confirmation",
        "noise_area_gate_plus_discrete_schedule",
    }:
        if dyn_cfg.noise_lookback not in context.noise_cache:
            context.noise_cache[dyn_cfg.noise_lookback] = compute_noise_sigma(context.minute_df, dyn_cfg.noise_lookback)
        noise_sigma = context.noise_cache[dyn_cfg.noise_lookback]

    dyn_mask = dynamic_gate_mask(
        candidate_df=candidate_df,
        config=dyn_cfg,
        noise_sigma=noise_sigma,
        atr_col=atr_col,
    )

    pass_mask = candidate_df["candidate_base_pass"].fillna(False).astype(bool)
    if experiment.compression.usage == "hard_filter":
        pass_mask &= comp_mask.fillna(False).astype(bool)
    pass_mask &= dyn_mask.fillna(False).astype(bool)

    selected_pre_ensemble = first_pass_signal_rows(candidate_df, pass_mask)
    thresholds = calibrate_ensemble_thresholds(
        selected_signal_rows=selected_pre_ensemble,
        is_sessions=context.is_sessions,
        atr_col=atr_col,
        q_lows_pct=ensemble.q_lows_pct,
        q_highs_pct=ensemble.q_highs_pct,
    )

    scored = apply_ensemble_selection(
        selected_signal_rows=selected_pre_ensemble,
        atr_col=atr_col,
        thresholds=thresholds,
        vote_threshold=ensemble.vote_threshold,
        compression_config=experiment.compression,
    )
    selected_final = scored.loc[scored.get("ensemble_selected", False)].copy()

    signal_df = build_signal_frame_for_backtest(context.minute_df, selected_final)
    trades = run_exit_variant_backtest(
        signal_df=signal_df,
        execution_model=ExecutionModel(),
        baseline=experiment.baseline_entry,
        exit_cfg=experiment.exit,
        max_leverage=max_leverage,
    )

    overall = compute_extended_metrics(
        trades=trades,
        signal_df=None,
        sessions=context.all_sessions,
        initial_capital=experiment.baseline_entry.account_size_usd,
        bootstrap_paths=bootstrap_paths,
        random_seed=random_seed,
    )

    is_set = set(pd.to_datetime(pd.Index(context.is_sessions)).date)
    oos_set = set(pd.to_datetime(pd.Index(context.oos_sessions)).date)
    trade_dates = pd.to_datetime(trades["session_date"]).dt.date if not trades.empty else pd.Series(dtype=object)
    trades_is = trades.loc[trade_dates.isin(is_set)].copy() if not trades.empty else trades.copy()
    trades_oos = trades.loc[trade_dates.isin(oos_set)].copy() if not trades.empty else trades.copy()

    metrics_is = compute_extended_metrics(
        trades=trades_is,
        signal_df=None,
        sessions=context.is_sessions,
        initial_capital=experiment.baseline_entry.account_size_usd,
        bootstrap_paths=max(300, bootstrap_paths // 2),
        random_seed=random_seed,
    )
    metrics_oos = compute_extended_metrics(
        trades=trades_oos,
        signal_df=None,
        sessions=context.oos_sessions,
        initial_capital=experiment.baseline_entry.account_size_usd,
        bootstrap_paths=max(300, bootstrap_paths // 2),
        random_seed=random_seed + 17,
    )

    score_overall = compute_scores(pd.Series(overall))
    score_is = compute_scores(pd.Series(metrics_is))
    score_oos = compute_scores(pd.Series(metrics_oos))

    row: dict[str, object] = {
        "name": experiment.name,
        "stage": experiment.stage,
        "family": experiment.family,
        "config_json": _serialize_config(experiment),
        "status": "ok",
        "candidate_rows_raw": int(candidate_df["candidate_base_pass"].sum()),
        "candidate_rows_after_overlays": int(pass_mask.sum()),
        "candidate_days_pre_ensemble": int(selected_pre_ensemble["session_date"].nunique()) if not selected_pre_ensemble.empty else 0,
        "selected_days": int(selected_final["session_date"].nunique()) if not selected_final.empty else 0,
        "atr_window": int(ensemble.atr_window),
        "ensemble_vote_threshold": float(ensemble.vote_threshold),
        "compression_mode": experiment.compression.mode,
        "compression_usage": experiment.compression.usage,
        "exit_mode": experiment.exit.mode,
        "dynamic_mode": experiment.dynamic_threshold.mode,
        "noise_lookback": int(experiment.dynamic_threshold.noise_lookback),
        "noise_vm": float(experiment.dynamic_threshold.noise_vm),
        "noise_k": float(experiment.dynamic_threshold.noise_k),
        "atr_k": float(experiment.dynamic_threshold.atr_k),
        "confirm_bars": int(experiment.dynamic_threshold.confirm_bars),
        "schedule": experiment.dynamic_threshold.schedule,
    }

    for prefix, metrics in (("overall", overall), ("is", metrics_is), ("oos", metrics_oos)):
        for key, value in metrics.items():
            row[f"{prefix}_{key}"] = value

    row.update(
        {
            "overall_academic_score": score_overall["academic_score"],
            "overall_prop_score": score_overall["prop_score"],
            "is_academic_score": score_is["academic_score"],
            "is_prop_score": score_is["prop_score"],
            "oos_academic_score": score_oos["academic_score"],
            "oos_prop_score": score_oos["prop_score"],
        }
    )

    detail = None
    if keep_details:
        detail = {
            "trades": trades,
            "selected_final": selected_final,
            "signal_df": signal_df,
        }
    return row, detail


In [4]:
SYMBOL = "MNQ"
INITIAL_CAPITAL_USD = 50_000.0
IS_FRACTION = 0.70
PLOT_TEMPLATE = "plotly_dark"

MNQ_1M_DATASET_PATH = None

ORB_RESEARCH_CONFIG_NAME = "full_reopt__seed__pair__comp_dynamic__weak_close__noise_area_gate"
ORB_OR_MINUTES = 15
ORB_OPENING_TIME = "09:30:00"
ORB_DIRECTION = "long"
ORB_ONE_TRADE_PER_DAY = True
ORB_ENTRY_BUFFER_TICKS = 2
ORB_STOP_BUFFER_TICKS = 2
ORB_TARGET_MULTIPLE = 2.0
ORB_VWAP_CONFIRMATION = True
ORB_VWAP_COLUMN = "continuous_session_vwap"
ORB_TIME_EXIT = "16:00:00"
ORB_RISK_PER_TRADE_PCT = 0.50
ORB_MAX_LEVERAGE = None

ORB_ATR_WINDOW = 14
ORB_Q_LOW_PCTS = (20, 25, 30)
ORB_Q_HIGH_PCTS = (90, 95)
ORB_ENSEMBLE_VOTE_THRESHOLD = 0.50
ORB_COMPRESSION_MODE = "weak_close"
ORB_COMPRESSION_USAGE = "soft_vote_bonus"
ORB_COMPRESSION_SOFT_BONUS_VOTES = 1.0
ORB_EXIT_MODE = "baseline"
ORB_DYNAMIC_MODE = "noise_area_gate"
ORB_NOISE_LOOKBACK = 30
ORB_NOISE_VM = 1.0
ORB_NOISE_K = 0.0
ORB_DYNAMIC_ATR_K = 0.0
ORB_DYNAMIC_CONFIRM_BARS = 1
ORB_DYNAMIC_SCHEDULE = "continuous_on_bar_close"
ORB_DYNAMIC_THRESHOLD_STYLE = "max_or_high_noise"

ORB_GRID_BOOTSTRAP_PATHS = 0
ORB_BASELINE_BOOTSTRAP_PATHS = 300

ORB_OR_MINUTES_GRID = (10, 15, 20, 30)
ORB_TARGET_MULTIPLE_GRID = (1.5, 2.0, 2.5, 3.0)
ORB_ENTRY_BUFFER_TICKS_GRID = (0, 1, 2, 3)
ORB_STOP_BUFFER_TICKS_GRID = (0, 1, 2, 3)
ORB_VWAP_CONFIRMATION_GRID = (False, True)
ORB_TIME_EXIT_GRID = ("15:30:00", "16:00:00")
ORB_RISK_PER_TRADE_PCT_GRID = (0.25, 0.50, 0.75, 1.00)
ORB_COMPRESSION_MODE_GRID = ("none", "weak_close", "strong_close", "nr4", "nr7", "inside_day")
ORB_COMPRESSION_USAGE_GRID = ("hard_filter", "soft_vote_bonus")
ORB_NOISE_LOOKBACK_GRID = (10, 14, 20, 30)
ORB_NOISE_VM_GRID = (0.75, 1.0, 1.25, 1.5)
ORB_ATR_WINDOW_GRID = (10, 14, 20, 30)
ORB_VOTE_THRESHOLD_GRID = (0.50, 0.67, 0.75)
ORB_QUANTILE_SET_GRID = (
    ((20,), (90,)),
    ((20,), (95,)),
    ((25,), (95,)),
    ((20, 25), (90, 95)),
    ((20, 25, 30), (90, 95)),
)
ORB_DYNAMIC_MODE_GRID = ("disabled", "noise_area_gate", "atr_threshold_gate", "close_confirmation_gate")
ORB_ATR_K_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)
ORB_CONFIRM_BARS_GRID = (1, 2, 3)

dataset_path = resolve_dataset_path(MNQ_1M_DATASET_PATH, SYMBOL, timeframe="1m")
instrument_spec = get_instrument_spec(SYMBOL)

parameter_rows = [
    {"section": "global", "parameter": "SYMBOL", "value": SYMBOL, "meaning": "Contrat analyse."},
    {"section": "global", "parameter": "INITIAL_CAPITAL_USD", "value": INITIAL_CAPITAL_USD, "meaning": "Capital de reference pour la courbe d'equity et le sizing."},
    {"section": "global", "parameter": "IS_FRACTION", "value": IS_FRACTION, "meaning": "Part historique reservee a l'in-sample."},
    {"section": "data", "parameter": "MNQ_1M_DATASET_PATH", "value": str(dataset_path), "meaning": "Dataset minute recharge a chaque execution."},
    {"section": "entry", "parameter": "ORB_OR_MINUTES", "value": ORB_OR_MINUTES, "meaning": "Longueur de l'opening range."},
    {"section": "entry", "parameter": "ORB_DIRECTION", "value": ORB_DIRECTION, "meaning": "Direction retenue: long only."},
    {"section": "entry", "parameter": "ORB_ENTRY_BUFFER_TICKS", "value": ORB_ENTRY_BUFFER_TICKS, "meaning": "Buffer a casser au-dessus du range."},
    {"section": "entry", "parameter": "ORB_STOP_BUFFER_TICKS", "value": ORB_STOP_BUFFER_TICKS, "meaning": "Buffer ajoute sous le plus bas OR pour le stop."},
    {"section": "entry", "parameter": "ORB_TARGET_MULTIPLE", "value": ORB_TARGET_MULTIPLE, "meaning": "Target en multiple du risque initial."},
    {"section": "entry", "parameter": "ORB_VWAP_CONFIRMATION", "value": ORB_VWAP_CONFIRMATION, "meaning": "Confirmation par VWAP continu."},
    {"section": "entry", "parameter": "ORB_TIME_EXIT", "value": ORB_TIME_EXIT, "meaning": "Heure de sortie forcee."},
    {"section": "risk", "parameter": "ORB_RISK_PER_TRADE_PCT", "value": ORB_RISK_PER_TRADE_PCT, "meaning": "Risque par trade en % du capital."},
    {"section": "risk", "parameter": "ORB_MAX_LEVERAGE", "value": ORB_MAX_LEVERAGE, "meaning": "Cap notionnel optionnel."},
    {"section": "ensemble", "parameter": "ORB_ATR_WINDOW", "value": ORB_ATR_WINDOW, "meaning": "Fenetre ATR du filtre d'ensemble."},
    {"section": "ensemble", "parameter": "ORB_Q_LOW_PCTS / ORB_Q_HIGH_PCTS", "value": f"{ORB_Q_LOW_PCTS} / {ORB_Q_HIGH_PCTS}", "meaning": "Bandes de quantiles ATR qui votent pour retenir une session."},
    {"section": "ensemble", "parameter": "ORB_ENSEMBLE_VOTE_THRESHOLD", "value": ORB_ENSEMBLE_VOTE_THRESHOLD, "meaning": "Seuil de vote minimum."},
    {"section": "overlay", "parameter": "ORB_COMPRESSION_MODE / USAGE", "value": f"{ORB_COMPRESSION_MODE} / {ORB_COMPRESSION_USAGE}", "meaning": "Overlay daily retenu."},
    {"section": "overlay", "parameter": "ORB_DYNAMIC_MODE", "value": ORB_DYNAMIC_MODE, "meaning": "Famille de gate dynamique retenue."},
    {"section": "overlay", "parameter": "ORB_NOISE_LOOKBACK / ORB_NOISE_VM", "value": f"{ORB_NOISE_LOOKBACK} / {ORB_NOISE_VM}", "meaning": "Parametres du noise gate."},
]
display(Markdown("## 0. Parametrage client"))
params = parameter_table(parameter_rows)


## 0. Parametrage client

,section,parameter,value,meaning
0,global,SYMBOL,MNQ,Contrat analyse.
1,global,INITIAL_CAPITAL_USD,50000.0,Capital de reference pour la courbe d'equity e...
2,global,IS_FRACTION,0.7,Part historique reservee a l'in-sample.
3,data,MNQ_1M_DATASET_PATH,D:\Business\Trading\VSCODE\algo-trading-intrad...,Dataset minute recharge a chaque execution.
4,entry,ORB_OR_MINUTES,15,Longueur de l'opening range.
5,entry,ORB_DIRECTION,long,Direction retenue: long only.
6,entry,ORB_ENTRY_BUFFER_TICKS,2,Buffer a casser au-dessus du range.
7,entry,ORB_STOP_BUFFER_TICKS,2,Buffer ajoute sous le plus bas OR pour le stop.
8,entry,ORB_TARGET_MULTIPLE,2.0,Target en multiple du risque initial.
9,entry,ORB_VWAP_CONFIRMATION,True,Confirmation par VWAP continu.


In [5]:
raw_1m = load_symbol_data(SYMBOL, input_paths={SYMBOL: dataset_path})
raw_1m["timestamp"] = pd.to_datetime(raw_1m["timestamp"], errors="coerce")

display(Markdown("## 1. Data et sanity checks"))
data_sanity = pd.DataFrame(
    [
        {"item": "rows_1m", "value": f"{len(raw_1m):,}"},
        {"item": "first_timestamp", "value": str(raw_1m["timestamp"].min())},
        {"item": "last_timestamp", "value": str(raw_1m["timestamp"].max())},
        {"item": "duplicate_timestamps", "value": int(raw_1m["timestamp"].duplicated().sum())},
        {"item": "tick_size", "value": instrument_spec["tick_size"]},
        {"item": "tick_value_usd", "value": instrument_spec["tick_value_usd"]},
        {"item": "point_value_usd", "value": instrument_spec["point_value_usd"]},
        {"item": "commission_per_side_usd", "value": instrument_spec["commission_per_side_usd"]},
        {"item": "slippage_ticks", "value": instrument_spec["slippage_ticks"]},
    ]
)
display(data_sanity)

orb_context_cache = {}

def make_orb_entry(**overrides):
    payload = {
        "or_minutes": ORB_OR_MINUTES,
        "opening_time": ORB_OPENING_TIME,
        "direction": ORB_DIRECTION,
        "one_trade_per_day": ORB_ONE_TRADE_PER_DAY,
        "entry_buffer_ticks": ORB_ENTRY_BUFFER_TICKS,
        "stop_buffer_ticks": ORB_STOP_BUFFER_TICKS,
        "target_multiple": ORB_TARGET_MULTIPLE,
        "vwap_confirmation": ORB_VWAP_CONFIRMATION,
        "vwap_column": ORB_VWAP_COLUMN,
        "time_exit": ORB_TIME_EXIT,
        "account_size_usd": INITIAL_CAPITAL_USD,
        "risk_per_trade_pct": ORB_RISK_PER_TRADE_PCT,
        "tick_size": float(instrument_spec["tick_size"]),
        "entry_on_next_open": True,
    }
    payload.update(overrides)
    return BaselineEntryConfig(**payload)

def make_orb_ensemble(**overrides):
    payload = {
        "atr_window": ORB_ATR_WINDOW,
        "q_lows_pct": tuple(ORB_Q_LOW_PCTS),
        "q_highs_pct": tuple(ORB_Q_HIGH_PCTS),
        "vote_threshold": ORB_ENSEMBLE_VOTE_THRESHOLD,
    }
    payload.update(overrides)
    return BaselineEnsembleConfig(**payload)

def make_orb_compression(**overrides):
    payload = {
        "mode": ORB_COMPRESSION_MODE,
        "usage": ORB_COMPRESSION_USAGE,
        "soft_bonus_votes": ORB_COMPRESSION_SOFT_BONUS_VOTES,
    }
    payload.update(overrides)
    return CompressionConfig(**payload)

def make_orb_dynamic(**overrides):
    payload = {
        "mode": ORB_DYNAMIC_MODE,
        "noise_lookback": ORB_NOISE_LOOKBACK,
        "noise_vm": ORB_NOISE_VM,
        "threshold_style": ORB_DYNAMIC_THRESHOLD_STYLE,
        "noise_k": ORB_NOISE_K,
        "atr_k": ORB_DYNAMIC_ATR_K,
        "confirm_bars": ORB_DYNAMIC_CONFIRM_BARS,
        "schedule": ORB_DYNAMIC_SCHEDULE,
    }
    payload.update(overrides)
    return DynamicThresholdConfig(**payload)

def make_orb_exit(**overrides):
    payload = {"mode": ORB_EXIT_MODE}
    payload.update(overrides)
    return ExitConfig(**payload)

def make_orb_experiment(entry=None, ensemble=None, compression=None, dynamic=None, exit_cfg=None, name=None):
    return ExperimentConfig(
        name=name or ORB_RESEARCH_CONFIG_NAME,
        stage="full_reopt",
        family="full_reopt",
        baseline_entry=entry or make_orb_entry(),
        baseline_ensemble=ensemble or make_orb_ensemble(),
        compression=compression or make_orb_compression(),
        exit=exit_cfg or make_orb_exit(),
        dynamic_threshold=dynamic or make_orb_dynamic(),
    )

def orb_context_key(entry_cfg, atr_windows):
    return (
        int(entry_cfg.or_minutes),
        str(entry_cfg.opening_time),
        str(entry_cfg.direction),
        int(entry_cfg.entry_buffer_ticks),
        bool(entry_cfg.vwap_confirmation),
        str(entry_cfg.vwap_column),
        tuple(sorted(int(x) for x in atr_windows)),
    )

def get_orb_context(entry_cfg, atr_windows):
    key = orb_context_key(entry_cfg, atr_windows)
    if key not in orb_context_cache:
        minute_df = prepare_minute_dataset(dataset_path=dataset_path, baseline_entry=entry_cfg, atr_windows=atr_windows)
        daily_reference = build_daily_reference(minute_df)
        minute_df = attach_daily_reference(minute_df, daily_reference)
        candidate_base = build_candidate_universe(minute_df, baseline_entry=entry_cfg)
        all_sessions = sorted(pd.to_datetime(minute_df["session_date"]).dt.date.unique())
        is_sessions, oos_sessions = _split_sessions(all_sessions, IS_FRACTION)
        orb_context_cache[key] = CampaignContext(
            all_sessions=all_sessions,
            is_sessions=is_sessions,
            oos_sessions=oos_sessions,
            minute_df=minute_df,
            candidate_base_df=candidate_base,
            daily_patterns=daily_reference,
        )
    return orb_context_cache[key]

def evaluate_orb_experiment_local(experiment, keep_details=False, bootstrap_paths=0, random_seed=42, max_leverage=None):
    required_windows = sorted({int(experiment.baseline_ensemble.atr_window), *[int(x) for x in ORB_ATR_WINDOW_GRID]})
    context = get_orb_context(experiment.baseline_entry, required_windows)
    return _evaluate_experiment(
        experiment=experiment,
        context=context,
        bootstrap_paths=bootstrap_paths,
        random_seed=random_seed,
        keep_details=keep_details,
        max_leverage=max_leverage,
    ), context

retained_experiment = make_orb_experiment()
(retained_row, retained_detail), retained_context = evaluate_orb_experiment_local(
    retained_experiment,
    keep_details=True,
    bootstrap_paths=ORB_BASELINE_BOOTSTRAP_PATHS,
    random_seed=42,
    max_leverage=ORB_MAX_LEVERAGE,
)

if retained_detail is None:
    raise RuntimeError(f"Retained ORB evaluation failed: {retained_row}")

orb_trades = retained_detail["trades"].copy()
orb_signal_df = retained_detail["signal_df"].copy()
orb_selected_final = retained_detail["selected_final"].copy()
orb_all_sessions = normalize_sessions(retained_context.all_sessions)
orb_is_sessions = normalize_sessions(retained_context.is_sessions)
orb_oos_sessions = normalize_sessions(retained_context.oos_sessions)
orb_daily_full = daily_results_from_trades(orb_trades, orb_all_sessions, INITIAL_CAPITAL_USD)
orb_summary = summarize_strategy_scopes("ORB retained", orb_daily_full, orb_trades, orb_is_sessions, orb_oos_sessions, INITIAL_CAPITAL_USD)

display(Markdown("## 2. Reconstruction de la strategie retenue"))
display(Markdown(
    "Config retenue: `full_reopt__seed__pair__comp_dynamic__weak_close__noise_area_gate` "
    "avec OR15 long-only, vote ATR, bonus `weak_close` et `noise_area_gate`."
))
display(pd.DataFrame(
    [
        {"item": "candidate_rows_raw", "value": int(retained_row.get("candidate_rows_raw", 0))},
        {"item": "candidate_rows_after_overlays", "value": int(retained_row.get("candidate_rows_after_overlays", 0))},
        {"item": "selected_days_after_ensemble", "value": int(retained_row.get("selected_days", 0))},
        {"item": "full_trades", "value": int(len(orb_trades))},
        {"item": "oos_trades", "value": int(len(subset_trades(orb_trades, orb_oos_sessions)))},
    ]
))

config_rows = []
for block, payload in [
    ("entry", asdict(retained_experiment.baseline_entry)),
    ("ensemble", asdict(retained_experiment.baseline_ensemble)),
    ("compression", asdict(retained_experiment.compression)),
    ("dynamic_threshold", asdict(retained_experiment.dynamic_threshold)),
    ("exit", asdict(retained_experiment.exit)),
]:
    for parameter, value in payload.items():
        config_rows.append({"block": block, "parameter": parameter, "value": value})
display(pd.DataFrame(config_rows))
display(orb_summary.round(3))

orb_is_curve = orb_daily_full.loc[orb_daily_full["session_date"].isin(set(orb_is_sessions))].copy()
orb_oos_curve = daily_results_from_trades(subset_trades(orb_trades, orb_oos_sessions), orb_oos_sessions, INITIAL_CAPITAL_USD)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Full sample", "OOS rebased"))
fig.add_trace(go.Scatter(x=orb_daily_full["session_date"], y=orb_daily_full["equity"], mode="lines", name="ORB full", line=dict(color="#2563eb", width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=orb_oos_curve["session_date"], y=orb_oos_curve["equity"], mode="lines", name="ORB OOS", line=dict(color="#16a34a", width=2.5)), row=1, col=2)
fig.update_layout(template=PLOT_TEMPLATE, width=1350, height=500, legend=dict(orientation="h", y=-0.15))
fig.update_yaxes(title_text="Equity USD", row=1, col=1)
fig.update_yaxes(title_text="Equity USD", row=1, col=2)
fig.show()

display(Markdown("### Extrait des trades"))
display(orb_trades.head(20))


## 1. Data et sanity checks

,item,value
0,rows_1m,"2,401,697"
1,first_timestamp,2019-05-05 18:03:00-04:00
2,last_timestamp,2026-03-20 09:29:00-04:00
3,duplicate_timestamps,0
4,tick_size,0.25
5,tick_value_usd,0.5
6,point_value_usd,2.0
7,commission_per_side_usd,1.25
8,slippage_ticks,1


## 2. Reconstruction de la strategie retenue

Config retenue: `full_reopt__seed__pair__comp_dynamic__weak_close__noise_area_gate` avec OR15 long-only, vote ATR, bonus `weak_close` et `noise_area_gate`.

,item,value
0,candidate_rows_raw,412288
1,candidate_rows_after_overlays,96219
2,selected_days_after_ensemble,444
3,full_trades,298
4,oos_trades,76


,block,parameter,value
0,entry,or_minutes,15
1,entry,opening_time,09:30:00
2,entry,direction,long
3,entry,one_trade_per_day,True
4,entry,entry_buffer_ticks,2
5,entry,stop_buffer_ticks,2
6,entry,target_multiple,2.0
7,entry,vwap_confirmation,True
8,entry,vwap_column,continuous_session_vwap
9,entry,time_exit,16:00:00


,strategy,scope,net_pnl_usd,return_pct,cagr_pct,sharpe,sortino,annualized_vol_pct,profit_factor,max_drawdown_usd,max_drawdown_pct,worst_day_usd,n_trades,win_rate
0,ORB retained,full,9776.0,19.552,2.631,0.883,0.398,2.411,1.407,1913.5,3.476,-249.0,298,0.527
1,ORB retained,is,4620.5,9.241,1.851,0.615,0.276,2.463,1.249,1913.5,3.476,-249.0,222,0.514
2,ORB retained,oos,5155.5,10.311,4.882,1.561,0.733,2.484,1.951,590.5,1.135,-247.5,76,0.566


### Extrait des trades

,trade_id,session_date,direction,quantity,entry_time,entry_price,stop_price,target_price,exit_time,exit_price,exit_reason,account_size_usd,risk_per_trade_pct,risk_budget_usd,risk_per_contract_usd,actual_risk_usd,trade_risk_usd,notional_usd,leverage_used,pnl_points,pnl_ticks,pnl_usd,fees,net_pnl_usd
0,1,2019-06-18,long,1,2019-06-18 09:49:00-04:00,7678.50,7597.50,7840.50,2019-06-18 16:00:00-04:00,7638.25,time_exit,50000.0,0.5,250.0,165.0,165.0,165.0,15357.0,0.30714,-40.25,-161.0,-80.5,2.5,-83.0
1,2,2019-08-06,long,2,2019-08-06 10:04:00-04:00,7530.50,7489.25,7613.00,2019-08-06 10:14:00-04:00,7489.00,stop,50000.0,0.5,250.0,85.5,171.0,171.0,30122.0,0.60244,-41.50,-166.0,-166.0,5.0,-171.0
2,3,2019-08-08,long,2,2019-08-08 10:01:00-04:00,7624.50,7583.00,7707.50,2019-08-08 14:42:00-04:00,7707.25,target,50000.0,0.5,250.0,86.0,172.0,172.0,30498.0,0.60996,82.75,331.0,331.0,5.0,326.0
3,4,2019-08-13,long,1,2019-08-13 09:46:00-04:00,7655.50,7554.75,7857.00,2019-08-13 16:00:00-04:00,7740.25,time_exit,50000.0,0.5,250.0,204.5,204.5,204.5,15311.0,0.30622,84.75,339.0,169.5,2.5,167.0
4,5,2019-09-03,long,2,2019-09-03 09:52:00-04:00,7670.50,7627.00,7757.50,2019-09-03 10:04:00-04:00,7626.75,stop,50000.0,0.5,250.0,90.0,180.0,180.0,30682.0,0.61364,-43.75,-175.0,-175.0,5.0,-180.0
5,6,2019-09-12,long,2,2019-09-12 10:11:00-04:00,7972.00,7924.00,8068.00,2019-09-12 10:51:00-04:00,7923.75,stop,50000.0,0.5,250.0,99.0,198.0,198.0,31888.0,0.63776,-48.25,-193.0,-193.0,5.0,-198.0
6,7,2019-10-03,long,2,2019-10-03 11:09:00-04:00,7600.75,7548.50,7705.25,2019-10-03 16:00:00-04:00,7655.75,time_exit,50000.0,0.5,250.0,107.5,215.0,215.0,30403.0,0.60806,55.00,220.0,220.0,5.0,215.0
7,8,2019-12-13,long,2,2019-12-13 10:02:00-05:00,8493.25,8439.50,8600.75,2019-12-13 16:00:00-05:00,8490.75,time_exit,50000.0,0.5,250.0,110.5,221.0,221.0,33973.0,0.67946,-2.50,-10.0,-10.0,5.0,-15.0
8,9,2020-02-03,long,1,2020-02-03 09:46:00-05:00,9108.25,9035.25,9254.25,2020-02-03 16:00:00-05:00,9128.25,time_exit,50000.0,0.5,250.0,149.0,149.0,149.0,18216.5,0.36433,20.00,80.0,40.0,2.5,37.5
9,10,2020-03-04,long,1,2020-03-04 10:23:00-05:00,8783.25,8686.75,8976.25,2020-03-04 16:00:00-05:00,8917.00,time_exit,50000.0,0.5,250.0,196.0,196.0,196.0,17566.5,0.35133,133.75,535.0,267.5,2.5,265.0


In [6]:
display(Markdown("## 3. Heatmaps IS/OOS - Parametres d'entree, execution et risque"))

entry_target_rows = []
counter = 0
for or_minutes in ORB_OR_MINUTES_GRID:
    for target_multiple in ORB_TARGET_MULTIPLE_GRID:
        counter += 1
        exp = make_orb_experiment(
            entry=make_orb_entry(or_minutes=int(or_minutes), target_multiple=float(target_multiple)),
            name=f"or{or_minutes}_tm{target_multiple}",
        )
        (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
        entry_target_rows.extend(row_to_scope_records(row, {"or_minutes": int(or_minutes), "target_multiple": float(target_multiple)}))
orb_or_target_grid = pd.DataFrame(entry_target_rows)
plot_is_oos_heatmaps(orb_or_target_grid, "target_multiple", "or_minutes", "sharpe", "ORB | OR minutes x target multiple | Sharpe")
plot_is_oos_heatmaps(orb_or_target_grid, "target_multiple", "or_minutes", "net_pnl_usd", "ORB | OR minutes x target multiple | Net PnL", text_auto=".0f")

buffer_rows = []
counter = 1_000
for entry_buffer in ORB_ENTRY_BUFFER_TICKS_GRID:
    for stop_buffer in ORB_STOP_BUFFER_TICKS_GRID:
        counter += 1
        exp = make_orb_experiment(
            entry=make_orb_entry(entry_buffer_ticks=int(entry_buffer), stop_buffer_ticks=int(stop_buffer)),
            name=f"eb{entry_buffer}_sb{stop_buffer}",
        )
        (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
        buffer_rows.extend(row_to_scope_records(row, {"entry_buffer_ticks": int(entry_buffer), "stop_buffer_ticks": int(stop_buffer)}))
orb_buffer_grid = pd.DataFrame(buffer_rows)
plot_is_oos_heatmaps(orb_buffer_grid, "entry_buffer_ticks", "stop_buffer_ticks", "sharpe", "ORB | entry buffer x stop buffer | Sharpe")
plot_is_oos_heatmaps(orb_buffer_grid, "entry_buffer_ticks", "stop_buffer_ticks", "max_drawdown_usd", "ORB | entry buffer x stop buffer | Max DD", text_auto=".0f", color_continuous_scale="RdYlGn_r")

vwap_time_rows = []
counter = 2_000
for vwap_confirmation in ORB_VWAP_CONFIRMATION_GRID:
    for time_exit in ORB_TIME_EXIT_GRID:
        counter += 1
        exp = make_orb_experiment(
            entry=make_orb_entry(vwap_confirmation=bool(vwap_confirmation), time_exit=str(time_exit)),
            name=f"vwap{int(bool(vwap_confirmation))}_tx{time_exit.replace(':','')}",
        )
        (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
        vwap_time_rows.extend(row_to_scope_records(row, {"vwap_confirmation": str(bool(vwap_confirmation)), "time_exit": str(time_exit)}))
orb_vwap_time_grid = pd.DataFrame(vwap_time_rows)
plot_is_oos_heatmaps(orb_vwap_time_grid, "time_exit", "vwap_confirmation", "sharpe", "ORB | VWAP confirmation x time exit | Sharpe")

risk_rows = []
counter = 3_000
for risk_pct in ORB_RISK_PER_TRADE_PCT_GRID:
    counter += 1
    exp = make_orb_experiment(entry=make_orb_entry(risk_per_trade_pct=float(risk_pct)), name=f"risk_{risk_pct}")
    (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
    risk_rows.extend(row_to_scope_records(row, {"risk_pct": float(risk_pct)}))
orb_risk_grid = pd.DataFrame(risk_rows)
plot_scope_heatmap(orb_risk_grid, "risk_pct", "sharpe", "ORB | risk per trade % | Sharpe")
plot_scope_heatmap(orb_risk_grid, "risk_pct", "max_drawdown_usd", "ORB | risk per trade % | Max DD", text_auto=".0f", color_continuous_scale="RdYlGn_r")


## 3. Heatmaps IS/OOS - Parametres d'entree, execution et risque

In [7]:
display(Markdown("## 4. Heatmaps IS/OOS - Overlays, dynamique et ensemble"))

compression_rows = []
counter = 4_000
for compression_mode in ORB_COMPRESSION_MODE_GRID:
    for compression_usage in ORB_COMPRESSION_USAGE_GRID:
        counter += 1
        exp = make_orb_experiment(
            compression=make_orb_compression(mode=str(compression_mode), usage=str(compression_usage)),
            name=f"comp_{compression_mode}_{compression_usage}",
        )
        (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
        compression_rows.extend(row_to_scope_records(row, {"compression_mode": str(compression_mode), "compression_usage": str(compression_usage)}))
orb_compression_grid = pd.DataFrame(compression_rows)
plot_is_oos_heatmaps(orb_compression_grid, "compression_usage", "compression_mode", "sharpe", "ORB | compression mode x usage | Sharpe")

noise_rows = []
counter = 5_000
for lookback in ORB_NOISE_LOOKBACK_GRID:
    for vm in ORB_NOISE_VM_GRID:
        counter += 1
        exp = make_orb_experiment(
            dynamic=make_orb_dynamic(mode="noise_area_gate", noise_lookback=int(lookback), noise_vm=float(vm), threshold_style="max_or_high_noise"),
            name=f"noise_l{lookback}_vm{vm}",
        )
        (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
        noise_rows.extend(row_to_scope_records(row, {"noise_lookback": int(lookback), "noise_vm": float(vm)}))
orb_noise_grid = pd.DataFrame(noise_rows)
plot_is_oos_heatmaps(orb_noise_grid, "noise_vm", "noise_lookback", "sharpe", "ORB | noise lookback x noise VM | Sharpe")
plot_is_oos_heatmaps(orb_noise_grid, "noise_vm", "noise_lookback", "max_drawdown_usd", "ORB | noise lookback x noise VM | Max DD", text_auto=".0f", color_continuous_scale="RdYlGn_r")

ensemble_rows = []
counter = 6_000
for atr_window in ORB_ATR_WINDOW_GRID:
    for vote_threshold in ORB_VOTE_THRESHOLD_GRID:
        counter += 1
        exp = make_orb_experiment(
            ensemble=make_orb_ensemble(atr_window=int(atr_window), vote_threshold=float(vote_threshold)),
            name=f"atr{atr_window}_vote{vote_threshold}",
        )
        (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
        ensemble_rows.extend(row_to_scope_records(row, {"atr_window": int(atr_window), "vote_threshold": float(vote_threshold)}))
orb_ensemble_grid = pd.DataFrame(ensemble_rows)
plot_is_oos_heatmaps(orb_ensemble_grid, "vote_threshold", "atr_window", "sharpe", "ORB | ATR window x vote threshold | Sharpe")

quantile_rows = []
counter = 7_000
for q_lows, q_highs in ORB_QUANTILE_SET_GRID:
    counter += 1
    label = f"low{list(q_lows)}_high{list(q_highs)}"
    exp = make_orb_experiment(
        ensemble=make_orb_ensemble(q_lows_pct=tuple(q_lows), q_highs_pct=tuple(q_highs)),
        name=f"quantiles_{counter}",
    )
    (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
    quantile_rows.extend(row_to_scope_records(row, {"quantile_set": label}))
orb_quantile_grid = pd.DataFrame(quantile_rows)
plot_scope_heatmap(orb_quantile_grid, "quantile_set", "sharpe", "ORB | quantile-set alternatives | Sharpe")

dynamic_family_rows = []
counter = 8_000
for mode in ORB_DYNAMIC_MODE_GRID:
    counter += 1
    if mode == "noise_area_gate":
        dynamic_cfg = make_orb_dynamic(mode=mode, noise_lookback=ORB_NOISE_LOOKBACK, noise_vm=ORB_NOISE_VM, threshold_style="max_or_high_noise")
    elif mode == "atr_threshold_gate":
        dynamic_cfg = make_orb_dynamic(mode=mode, atr_k=0.5)
    elif mode == "close_confirmation_gate":
        dynamic_cfg = make_orb_dynamic(mode=mode, confirm_bars=1)
    else:
        dynamic_cfg = make_orb_dynamic(mode=mode)
    exp = make_orb_experiment(dynamic=dynamic_cfg, name=f"dynamic_{mode}")
    (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
    dynamic_family_rows.extend(row_to_scope_records(row, {"dynamic_mode": str(mode)}))
orb_dynamic_family_grid = pd.DataFrame(dynamic_family_rows)
plot_scope_heatmap(orb_dynamic_family_grid, "dynamic_mode", "sharpe", "ORB | dynamic mode family | Sharpe")

atrk_rows = []
counter = 9_000
for atr_k in ORB_ATR_K_GRID:
    counter += 1
    exp = make_orb_experiment(
        dynamic=make_orb_dynamic(mode="atr_threshold_gate", atr_k=float(atr_k)),
        name=f"atrk_{atr_k}",
    )
    (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
    atrk_rows.extend(row_to_scope_records(row, {"atr_k": float(atr_k)}))
orb_atrk_grid = pd.DataFrame(atrk_rows)
plot_scope_heatmap(orb_atrk_grid, "atr_k", "sharpe", "ORB | ATR threshold gate | Sharpe")

confirm_rows = []
counter = 10_000
for confirm_bars in ORB_CONFIRM_BARS_GRID:
    counter += 1
    exp = make_orb_experiment(
        dynamic=make_orb_dynamic(mode="close_confirmation_gate", confirm_bars=int(confirm_bars)),
        name=f"confirm_{confirm_bars}",
    )
    (row, _), _ = evaluate_orb_experiment_local(exp, keep_details=False, bootstrap_paths=ORB_GRID_BOOTSTRAP_PATHS, random_seed=100 + counter, max_leverage=ORB_MAX_LEVERAGE)
    confirm_rows.extend(row_to_scope_records(row, {"confirm_bars": int(confirm_bars)}))
orb_confirm_grid = pd.DataFrame(confirm_rows)
plot_scope_heatmap(orb_confirm_grid, "confirm_bars", "sharpe", "ORB | close confirmation bars | Sharpe")


## 4. Heatmaps IS/OOS - Overlays, dynamique et ensemble

In [8]:
display(Markdown("## 5. Lecture finale"))
oos_row = orb_summary.loc[orb_summary["scope"].eq("oos")].iloc[0]
notes = [
    f"- ORB retenue OOS: net `{fmt_money(oos_row['net_pnl_usd'])}`, Sharpe `{fmt_float(oos_row['sharpe'])}`, maxDD `{fmt_money(oos_row['max_drawdown_usd'])}`.",
    "- Les heatmaps se lisent toujours en IS/OOS : l'objectif n'est pas seulement le meilleur point, mais la zone stable.",
    "- Les premieres zones a challenger si le client veut adapter la sleeve sont : `noise_lookback / noise_vm`, `atr_window / vote_threshold`, puis `or_minutes / target_multiple`.",
    "- Le notebook est autonome : toute la logique de signal, d'overlay et de backtest est visible dans les cellules de code ci-dessus.",
]
display(Markdown("\n".join(notes)))


## 5. Lecture finale

- ORB retenue OOS: net `5,155.5 USD`, Sharpe `1.561`, maxDD `590.5 USD`.
- Les heatmaps se lisent toujours en IS/OOS : l'objectif n'est pas seulement le meilleur point, mais la zone stable.
- Les premieres zones a challenger si le client veut adapter la sleeve sont : `noise_lookback / noise_vm`, `atr_window / vote_threshold`, puis `or_minutes / target_multiple`.
- Le notebook est autonome : toute la logique de signal, d'overlay et de backtest est visible dans les cellules de code ci-dessus.